# Loading the data

In [4]:
import pandas as pd
from io import StringIO

# ── Helper for xlsx files with metadata header (data starts at row 5) ─────────
def load_xlsx(filename):
    df = pd.read_excel(filename, skiprows=4, header=0, usecols=[0, 1])
    df.columns = ["Date", "Value"]
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    return df.dropna(subset=["Date", "Value"]).reset_index(drop=True)

# ── CSVs (FRED-style) ─────────────────────────────────────────────────────────
yield_curve      = pd.read_csv("10Y-2Y.csv").rename(columns={"observation_date": "Date", "T10Y2Y": "Value"})
bbb_spread       = pd.read_csv("BBB-treasury_credit_spread.csv").rename(columns={"observation_date": "Date", "BAMLC0A4CBBB": "Value"})
buffett_csv      = pd.read_csv("Buffet_indicator.csv").rename(columns={"observation_date": "Date", "DDDM01USA156NWDB": "Value"})
erp              = pd.read_csv("ERP.csv").rename(columns={"observation_date": "Date", "TENEXPCHAREARISPRE": "Value"})
fear_greed       = pd.read_csv("fear_greed_indicator.csv").rename(columns={"Fear Greed": "Value"})
financial_stress = pd.read_csv("financial_stress_index.csv").rename(columns={"observation_date": "Date", "STLFSI4": "Value"})
misery           = pd.read_csv("Misery_index_urban_USA.csv").rename(columns={"observation_date": "Date", "UNRATE_CPIAUCSL_PC1": "Value"})
sahm             = pd.read_csv("Sahm_indicator.csv").rename(columns={"observation_date": "Date", "SAHMREALTIME": "Value"})
vix              = pd.read_csv("VIX_data.csv").rename(columns={"observation_date": "Date", "VIXCLS": "Value"})

# OFR FSI — quoted CSV, parse and extract both stress + equity valuation sub-index
with open("margin_debt_GDP.csv", "r") as f:
    content = f.read().replace('"', '')
ofr_raw          = pd.read_csv(StringIO(content))
ofr_raw["Date"]  = pd.to_datetime(ofr_raw["Date"], errors="coerce")
ofr_raw          = ofr_raw.dropna(subset=["Date"])
ofr_fsi          = ofr_raw[["Date", "OFR FSI"]].rename(columns={"OFR FSI": "Value"})
equity_valuation = ofr_raw[["Date", "Equity valuation"]].rename(columns={"Equity valuation": "Value"})

# SKEW — OHLCV, keep Close only
skew = pd.read_csv("CBOE_SKEW_historical.csv", usecols=["Date", "Close"]).rename(columns={"Close": "Value"})
skew["Date"] = pd.to_datetime(skew["Date"], errors="coerce")
skew = skew.dropna()

# ── Shiller (special multi-header xls) ───────────────────────────────────────
shiller_raw   = pd.read_excel("shiller_data.xls", sheet_name="Data", skiprows=6).iloc[1:]
shiller_raw   = shiller_raw.rename(columns={"Unnamed: 0": "Date", "P/E10 or": "Value"})
shiller_ratio = shiller_raw[["Date", "Value"]].dropna().copy()

# Date is a float like 1871.01 — convert year+fraction to proper datetime
shiller_ratio["Date"] = shiller_ratio["Date"].astype(float)
shiller_ratio["Year"] = shiller_ratio["Date"].astype(int)
shiller_ratio["Month"] = ((shiller_ratio["Date"] % 1) * 12 + 1).round().astype(int).clip(1, 12)
shiller_ratio["Date"] = pd.to_datetime(shiller_ratio[["Year", "Month"]].assign(day=1))
shiller_ratio = shiller_ratio[["Date", "Value"]].dropna()

print(shiller_ratio.shape)
print(shiller_ratio.head(3))
print(shiller_ratio.tail(3))

# ── AAII Sentiment (multi-header xls, data starts row 4) ─────────────────────
sentiment = pd.read_excel("sentiment.xls", skiprows=3, header=0, usecols=[0, 1, 2, 3])
sentiment.columns = ["Date", "Bullish", "Neutral", "Bearish"]
sentiment["Date"] = pd.to_datetime(sentiment["Date"], errors="coerce")
sentiment = sentiment.dropna(subset=["Date"])

# ── Brent & Gold (first row is junk header, data from row 2) ─────────────────
brent = pd.read_excel("brent_price.xlsx", skiprows=1, header=0, usecols=[0, 1])
brent.columns = ["Date", "Value"]
brent["Date"] = pd.to_datetime(brent["Date"], errors="coerce")
brent = brent.dropna()

gold = pd.read_excel("gold_prices.xlsx", skiprows=1, header=0, usecols=[0, 1])
gold.columns = ["Date", "Value"]
gold["Date"] = pd.to_datetime(gold["Date"], errors="coerce")
gold = gold.dropna()

# ── Standard xlsx files ───────────────────────────────────────────────────────
tobin_q        = load_xlsx("Tobin Q 2026-02-13 19_56_32.xlsx")
pe_ratio       = load_xlsx("S&P 500 PE Ratio 2026-02-13 20_01_50.xlsx")
pb_ratio       = load_xlsx("S&P 500 Price to Book Value 2026-02-13 20_01_32.xlsx")
ps_ratio       = load_xlsx("S&P 500 Price to Sales 2026-02-13 20_01_40.xlsx")
earnings_yield = load_xlsx("S&P 500 Earnings Yield 2026-02-13 20_02_48.xlsx")
dividend_yield = load_xlsx("S&P 500 Dividend Yield 2026-02-13 20_17_09.xlsx")
corp_margin    = load_xlsx("Corporate Profit Margin (After Tax)  2026-02-13 19_59_45.xlsx")
mktcap_gdp     = load_xlsx("USA Ratio of Total Market Cap over GDP 2026-02-13 19_58_12.xlsx")
insider_ratio  = load_xlsx("Insider Buy_Sell Ratio - USA Overall Market 2026-02-13 20_10_59.xlsx")
unemployment   = load_xlsx("Civilian Unemployment Rate 2026-02-13 20_08_59.xlsx")
gdp_recession  = load_xlsx("GDP-Based Recession Indicator Index 2026-02-13 20_04_13.xlsx")

# ── Sanity check ──────────────────────────────────────────────────────────────
datasets = {
    "yield_curve": yield_curve, "bbb_spread": bbb_spread, "buffett_csv": buffett_csv,
    "erp": erp, "fear_greed": fear_greed, "financial_stress": financial_stress,
    "misery": misery, "sahm": sahm, "vix": vix,
    "ofr_fsi": ofr_fsi, "equity_valuation": equity_valuation,
    "skew": skew, "shiller_ratio": shiller_ratio, "sentiment": sentiment,
    "brent": brent, "gold": gold, "tobin_q": tobin_q, "pe_ratio": pe_ratio,
    "pb_ratio": pb_ratio, "ps_ratio": ps_ratio, "earnings_yield": earnings_yield,
    "dividend_yield": dividend_yield, "corp_margin": corp_margin,
    "mktcap_gdp": mktcap_gdp, "insider_ratio": insider_ratio,
    "unemployment": unemployment, "gdp_recession": gdp_recession,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape} | {df['Date'].min()} → {df['Date'].max()}")

/var/folders/gf/6fqf762133b8cfkrb2dlj0g40000gn/T/ipykernel_82174/4228307511.py:33: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  skew["Date"] = pd.to_datetime(skew["Date"], errors="coerce")


(1741, 2)
          Date      Value
121 1881-01-01  18.473952
122 1881-01-01  18.147258
123 1881-01-01  18.270119
           Date      Value
1859 2025-02-01  39.272363
1860 2025-02-01   39.89132
1861 2026-01-01  39.850757
yield_curve: (12949, 2) | 1976-06-01 → 2026-01-16
bbb_spread: (7676, 2) | 1996-12-31 → 2026-01-15
buffett_csv: (46, 2) | 1975-01-01 → 2020-01-01
erp: (529, 2) | 1982-01-01 → 2026-01-01
fear_greed: (5080, 2) | 2011-01-03 → 2026-01-16
financial_stress: (1672, 2) | 1993-12-31 → 2026-01-09
misery: (936, 2) | 1948-01-01 → 2025-12-01
sahm: (793, 2) | 1959-12-01 → 2025-12-01
vix: (9465, 2) | 1990-01-02 → 2026-04-13
ofr_fsi: (6597, 2) | 2000-01-03 00:00:00 → 2026-01-30 00:00:00
equity_valuation: (6597, 2) | 2000-01-03 00:00:00 → 2026-01-30 00:00:00
skew: (9037, 2) | 1990-01-02 00:00:00-05:00 → 2026-03-06 00:00:00-05:00
shiller_ratio: (1741, 2) | 1881-01-01 00:00:00 → 2026-01-01 00:00:00
sentiment: (2013, 4) | 1987-06-26 00:00:00 → 2026-02-12 00:00:00
brent: (8790, 2) | 1991-0

In [5]:
technicals = pd.read_csv("spx_technicals_corrected.csv", skiprows=2)
technicals.columns = ["Date", "Price", "SMA50", "SMA200", "Golden_Cross", "Death_Cross",
                       "RSI_14", "MACD", "MACD_Signal", "BB_Upper", "BB_Lower"]
technicals["Date"] = pd.to_datetime(technicals["Date"], errors="coerce")
technicals = technicals.dropna(subset=["Date"]).reset_index(drop=True)

for col in ["Price", "SMA50", "SMA200", "RSI_14", "MACD", "MACD_Signal", "BB_Upper", "BB_Lower"]:
    technicals[col] = pd.to_numeric(technicals[col], errors="coerce")

technicals["Golden_Cross"] = technicals["Golden_Cross"].astype(str).str.strip().str.lower() == "true"
technicals["Death_Cross"]  = technicals["Death_Cross"].astype(str).str.strip().str.lower() == "true"

golden_crosses = technicals[technicals["Golden_Cross"]][["Date"]].copy()
death_crosses  = technicals[technicals["Death_Cross"]][["Date"]].copy()

print(technicals.shape)
print(technicals.tail(3))
print(f"Golden crosses: {len(golden_crosses)}, Death crosses: {len(death_crosses)}")

(24661, 11)
            Date        Price        SMA50       SMA200  Golden_Cross  \
24658 2026-03-04  6869.500000  6905.295186  6574.284731         False   
24659 2026-03-05  6830.709961  6905.219385  6578.646382         False   
24660 2026-03-06  6740.020020  6902.449980  6582.528481         False   

       Death_Cross     RSI_14       MACD  MACD_Signal     BB_Upper  \
24658        False  43.924416  -9.880644    -6.240079  6976.786851   
24659        False  49.803798 -12.920707    -7.576205  6977.029438   
24660        False  42.114335 -22.389803   -10.538924  6987.134670   

          BB_Lower  
24658  6788.795083  
24659  6783.351470  
24660  6767.408250  
Golden crosses: 51, Death crosses: 50


In [6]:
#gold-brent ratio
gold_brent = pd.merge(gold, brent, on="Date", how="inner").iloc[1:].rename(columns={"XAU= (BID)":"gold_price", "BRT- (MID_PRICE)":"brent_price"})
gold_brent["gold_brent_ratio"] = gold_brent.Value_x/gold_brent.Value_y
'''#gold-wti ratio
gold_wti = pd.merge(gold_data, wti_data, on="Date", how="inner").iloc[1:].rename(columns={"XAU= (BID)":"gold_price", "WTC- (MID_PRICE)":"brent_price"})
gold_wti["gold_wti_ratio"] = gold_wti.gold_price/gold_wti.brent_price'''

gold_brent.head()

,Date,Value_x,Value_y,gold_brent_ratio
1,2026-01-15,4614.62,67.83,68.032139
2,2026-01-14,4620.48,69.58,66.405289
3,2026-01-13,4587.39,68.11,67.352665
4,2026-01-12,4593.49,66.16,69.430018
5,2026-01-09,4509.79,65.88,68.454614


In [7]:
yield_curve      = pd.read_csv("10Y-2Y.csv").rename(columns={"observation_date": "Date", "T10Y2Y": "Value"})
bbb_spread       = pd.read_csv("BBB-treasury_credit_spread.csv").rename(columns={"observation_date": "Date", "BAMLC0A4CBBB": "Value"})
buffett_csv      = pd.read_csv("Buffet_indicator.csv").rename(columns={"observation_date": "Date", "DDDM01USA156NWDB": "Value"})
erp              = pd.read_csv("ERP.csv").rename(columns={"observation_date": "Date", "TENEXPCHAREARISPRE": "Value"})
fear_greed       = pd.read_csv("fear_greed_indicator.csv").rename(columns={"Fear Greed": "Value"})
financial_stress = pd.read_csv("financial_stress_index.csv").rename(columns={"observation_date": "Date", "STLFSI4": "Value"})
misery           = pd.read_csv("Misery_index_urban_USA.csv").rename(columns={"observation_date": "Date", "UNRATE_CPIAUCSL_PC1": "Value"})
sahm             = pd.read_csv("Sahm_indicator.csv").rename(columns={"observation_date": "Date", "SAHMREALTIME": "Value"})
vix              = pd.read_csv("VIX_data.csv").rename(columns={"observation_date": "Date", "VIXCLS": "Value"})

In [8]:
def prep_indicator(df, value_col, label=None):
    out = df.copy()
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out[value_col] = pd.to_numeric(out[value_col], errors="coerce")
    out = out.dropna(subset=["Date", value_col]).sort_values("Date")
    # standardize value column name to "value" so downstream code is uniform
    out = out.rename(columns={value_col: "value"})
    out["label"] = label if label else value_col
    return out[["Date", "value", "label"]]

# --- IMPORTANT: set these to match your actual column names in each DF ---
# (Because after rename to Date, the value columns can differ by file)
yield_value_col   = [c for c in yield_curve.columns if c != "Date"][0]
fsi_value_col     = [c for c in financial_stress.columns if c != "Date"][0]
bbb_value_col     = [c for c in bbb_spread.columns if c != "Date"][0]
misery_value_col  = [c for c in misery.columns if c != "Date"][0]
buffett_value_col = [c for c in buffett_csv.columns if c != "Date"][0]
sahm_value_col    = [c for c in sahm.columns if c != "Date"][0]
vix_value_col     = [c for c in vix.columns if c != "Date"][0]

# Fear/Greed already has Date column? If not, rename it the same way.
# If it already has Date, leave it. If not, uncomment the next line and adjust:
# fear_greed = fear_greed.rename(columns={"your_date_col_here":"Date"})
fear_greed_value_col = [c for c in fear_greed.columns if c != "Date"][0]

# Shiller: you already made shiller_ratio with Date + shiller_ratio
shiller_ratio = shiller_ratio.copy()
shiller_ratio["Date"] = pd.to_datetime(shiller_ratio["Date"], errors="coerce")
shiller_ratio["shiller_ratio"] = pd.to_numeric(shiller_ratio["Value"], errors="coerce")
shiller_ratio = shiller_ratio.dropna(subset=["Date","Value"]).sort_values("Date")

# --- Build indicator map ---
indicators = {
    "Gold / Brent ratio": prep_indicator(gold_brent, "gold_brent_ratio", "Gold/Brent"),

    "VIX": prep_indicator(vix, vix_value_col, "VIX"),
    "Fear & Greed": prep_indicator(fear_greed, fear_greed_value_col, "Fear & Greed"),

    "Yield curve (10Y–2Y)": prep_indicator(yield_curve, yield_value_col, "10Y-2Y"),
    "BBB – Treasury spread": prep_indicator(bbb_spread, bbb_value_col, "BBB-Tsy"),
    "Financial Stress Index": prep_indicator(financial_stress, fsi_value_col, "FSI"),

    "Misery index": prep_indicator(misery, misery_value_col, "Misery"),
    "Buffett indicator": prep_indicator(buffett_csv, buffett_value_col, "Buffett"),
    "Sahm rule": prep_indicator(sahm, sahm_value_col, "Sahm"),

    "Shiller CAPE (P/E10)": prep_indicator(shiller_ratio.rename(columns={"Value":"value"}), "value", "CAPE"),
}

print("Indicators ready:", list(indicators.keys()))

Indicators ready: ['Gold / Brent ratio', 'VIX', 'Fear & Greed', 'Yield curve (10Y–2Y)', 'BBB – Treasury spread', 'Financial Stress Index', 'Misery index', 'Buffett indicator', 'Sahm rule', 'Shiller CAPE (P/E10)']


In [10]:
# ── Derived series ────────────────────────────────────────────────────────────
import numpy as np

# List of indicators where a LOW value indicates HIGH risk/froth
INVERSE_INDICATORS = [
    "Earnings Yield",
    "Dividend Yield",
    "Equity Risk Premium",
    "Yield Curve (10Y–2Y)",
    "Insider Buy/Sell",
    "AAII Bullish",
    "BB Lower",
    "SPX SMA200"
]

def make_indicator(df, value_col, indicator_name):
    out = df.copy()
    if "Date" not in out.columns:
        raise ValueError(f"'Date' column not found.")

    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out[value_col] = pd.to_numeric(out[value_col], errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=["Date", value_col]).sort_values("Date")
    out = out.drop_duplicates(subset=["Date"], keep="last")

    # Standardize column naming
    out = out.rename(columns={value_col: "value"})

    # Tag whether this indicator should be inverted for "Right Tail = Bad" logic
    out["inverted"] = indicator_name in INVERSE_INDICATORS

    return out[["Date", "value", "inverted"]]

# Define raw names to make mapping cleaner
indicator_configs = {
    # ── Valuation ─────────────────────────────────────────────────────────────
    "Shiller CAPE":              (shiller_ratio,  "Value"),
    "PE Ratio":                  (pe_ratio,        "Value"),
    "PB Ratio":                  (pb_ratio,        "Value"),
    "PS Ratio":                  (ps_ratio,        "Value"),
    "Earnings Yield":            (earnings_yield,  "Value"),
    "Dividend Yield":            (dividend_yield,  "Value"),
    "Tobin Q":                   (tobin_q,         "Value"),
    "Buffet Indicator":          (mktcap_gdp,      "Value"),
    "Corp Profit Margin":        (corp_margin,     "Value"),
    "Equity Risk Premium":       (erp,             "Value"),
    "Equity Valuation (OFR)":    (equity_valuation,"Value"),

    # ── Credit & Macro Stress ─────────────────────────────────────────────────
    "Yield Curve (10Y–2Y)":      (yield_curve,      "Value"),
    "BBB–Treasury Spread":       (bbb_spread,        "Value"),
    "Financial Stress (StL)":    (financial_stress,  "Value"),
    "OFR Financial Stress":      (ofr_fsi,           "Value"),
    "VIX":                       (vix,               "Value"),
    "CBOE SKEW":                 (skew,              "Value"),

    # ── Sentiment & Positioning ───────────────────────────────────────────────
    "Fear & Greed":              (fear_greed,        "Value"),
    "Insider Buy/Sell":          (insider_ratio,     "Value"),
    "AAII Bullish":              (sentiment.rename(columns={"Bullish": "Value"}), "Value"),
    "AAII Bearish":              (sentiment.rename(columns={"Bearish": "Value"}), "Value"),
    "AAII Bull-Bear Spread":     (sentiment.assign(Value=sentiment["Bullish"] - sentiment["Bearish"]), "Value"),

    # ── Labor & Recession ─────────────────────────────────────────────────────
    "Sahm Rule":                 (sahm,             "Value"),
    "Misery Index":              (misery,           "Value"),
    "GDP Recession Indicator":   (gdp_recession,    "Value"),

    # ── Commodities & Ratios ──────────────────────────────────────────────────
    "Gold / Brent Ratio":        (gold_brent,       "gold_brent_ratio"),
    "Gold":                      (gold,             "Value"),  # Raw Gold Series
    "Brent Oil":                 (brent,            "Value"),  # Raw Oil Series

    # ── Technicals (SPX) ─────────────────────────────────────────────────────
    "SPX SMA50":                 (technicals,       "SMA50"),
    "SPX SMA200":                (technicals,       "SMA200"),
    "RSI (14)":                  (technicals,       "RSI_14"),
    "MACD":                      (technicals,       "MACD"),
    "BB Upper":                  (technicals,       "BB_Upper"),
    "BB Lower":                  (technicals,       "BB_Lower"),
}

# Construct the final indicators dictionary
indicators = {name: make_indicator(df, col, name) for name, (df, col) in indicator_configs.items()}

In [11]:
import plotly.io as pio
pio.renderers.default = "colab"

SPX_DATE_COL  = "Date"
SPX_CLOSE_COL = "Price"

def prep_spx(spx: pd.DataFrame) -> pd.DataFrame:
    df = spx.copy()
    df[SPX_DATE_COL]  = pd.to_datetime(df[SPX_DATE_COL], errors="coerce")
    df[SPX_CLOSE_COL] = pd.to_numeric(df[SPX_CLOSE_COL], errors="coerce")
    df = df.dropna(subset=[SPX_DATE_COL, SPX_CLOSE_COL]).sort_values(SPX_DATE_COL)
    df = df.rename(columns={SPX_DATE_COL: "Date", SPX_CLOSE_COL: "spx_close"})
    df = df.drop_duplicates(subset=["Date"], keep="last")
    return df[["Date", "spx_close"]]

spx_c = prep_spx(technicals)

def build_dataset(spx_df: pd.DataFrame, ind_df: pd.DataFrame, ind_col: str, horizon_days: int) -> pd.DataFrame:
    spx_df = spx_df.sort_values("Date").reset_index(drop=True).copy()
    ind_df = ind_df.sort_values("Date").reset_index(drop=True).copy()
    ind_df = ind_df.drop_duplicates(subset=["Date"], keep="last")
    aligned = pd.merge_asof(ind_df, spx_df, on="Date", direction="backward")
    spx_df["spx_future_close"] = spx_df["spx_close"].shift(-horizon_days)
    spx_future = spx_df[["Date", "spx_future_close"]]
    aligned2 = pd.merge_asof(
        aligned.sort_values("Date"),
        spx_future.sort_values("Date"),
        on="Date",
        direction="backward"
    )
    aligned2["fwd_return"] = (aligned2["spx_future_close"] / aligned2["spx_close"]) - 1.0
    out = aligned2.dropna(subset=[ind_col, "spx_close", "spx_future_close", "fwd_return"]).copy()
    out = out.drop_duplicates(subset=["Date"], keep="last")
    return out

def quantile_table(df: pd.DataFrame, xcol: str, ycol: str, q: int = 5) -> pd.DataFrame:
    tmp = df[[xcol, ycol]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if tmp[xcol].nunique() < q:
        tmp["_rank"] = tmp[xcol].rank(method="average")
        tmp["bucket"] = pd.qcut(tmp["_rank"], q=q, duplicates="drop")
    else:
        tmp["bucket"] = pd.qcut(tmp[xcol], q=q, duplicates="drop")
    tab = (
        tmp.groupby("bucket", observed=True)[ycol]
           .agg(["count", "mean", "median", "min", "max"])
           .reset_index()
    )
    tab["bucket"] = tab["bucket"].astype(str)
    return tab

def plot_spx_with_crosses(technicals, golden_crosses, death_crosses):
    fig = go.Figure()

    # SPX price
    fig.add_trace(go.Scatter(
        x=technicals["Date"], y=technicals["Price"],
        name="SPX", line=dict(color="white", width=1.5)
    ))

    # SMA50 and SMA200
    fig.add_trace(go.Scatter(
        x=technicals["Date"], y=technicals["SMA50"],
        name="SMA 50", line=dict(color="orange", width=1, dash="dot")
    ))
    fig.add_trace(go.Scatter(
        x=technicals["Date"], y=technicals["SMA200"],
        name="SMA 200", line=dict(color="cyan", width=1, dash="dot")
    ))

    # Golden cross markers
    gc_prices = technicals[technicals["Golden_Cross"]][["Date", "Price"]]
    fig.add_trace(go.Scatter(
        x=gc_prices["Date"], y=gc_prices["Price"],
        mode="markers", name="Golden Cross",
        marker=dict(symbol="triangle-up", color="gold", size=12, line=dict(color="white", width=1))
    ))

    # Death cross markers
    dc_prices = technicals[technicals["Death_Cross"]][["Date", "Price"]]
    fig.add_trace(go.Scatter(
        x=dc_prices["Date"], y=dc_prices["Price"],
        mode="markers", name="Death Cross",
        marker=dict(symbol="triangle-down", color="red", size=12, line=dict(color="white", width=1))
    ))

    fig.update_layout(
        title="S&P 500 — Golden & Death Crosses",
        template="plotly_dark",
        xaxis_title="Date",
        yaxis_title="Price",
        hovermode="x unified",
        height=550
    )
    fig.show()

# Evaluation of the combinations

In [12]:
def evaluate_all_indicator_search(
    horizon_years=1.0,  # Locked to 1 Year as requested
    drawdown_threshold=0.20,
    recovery_threshold=0.05,
    min_spacing_days=126,
    max_combo_size=1,
    max_all_indicators_to_search=None
):
    """
    Exhaustively scans the indicator library specifically looking for
    High Recall and F1-Score over a 1-year prediction horizon.
    """
    start_time = time.time()
    all_names = sorted(indicators.keys())

    if max_all_indicators_to_search:
        all_names = all_names[:max_all_indicators_to_search]

    print(f"🔎 Scanning {len(all_names)} indicators for {drawdown_threshold*100}% downturns (1Y Horizon)...")

    results = []

    for name in all_names:
        try:
            # Generate the 1-year horizon dataset for this indicator
            ds_single, _ = make_downturn_prediction_dataset(
                selected=[name],
                horizon_years=horizon_years, # 1.0
                drawdown_threshold=drawdown_threshold,
                recovery_threshold=recovery_threshold,
                min_spacing_days=min_spacing_days
            )

            if f"{name}__pct" not in ds_single.columns:
                continue

            signal = ds_single[f"{name}__pct"].to_numpy()
            y_v = ds_single["target_downturn"].to_numpy()

            # Filter for sufficient history (at least 252 trading days / 1 year)
            valid = ~np.isnan(signal)
            if valid.sum() < 252: continue

            signal = signal[valid]
            y_v = y_v[valid]

            # Compute Metrics (Top 10% of readings as the 'Alert' threshold)
            metrics = compute_top_bucket_metrics(y_v, signal, top_frac=0.10)

            results.append({
                "Indicator": name,
                "Horizon": f"{horizon_years}Y",
                "Precision": metrics['precision'],
                "Recall": metrics['recall'],      # Now explicitly tracked
                "F1_Score": metrics['f1'],        # The balance of Prec/Rec
                "AUC": auc_from_scores(y_v, signal),
                "Total_Obs": len(y_v),
                "Crashes_Caught": int(metrics['recall'] * y_v.sum()),
                "Total_Crashes": int(y_v.sum())
            })

        except Exception:
            continue

    # Rank by F1_Score to find the best balanced 'Leads'
    ranked_df = pd.DataFrame(results).sort_values("F1_Score", ascending=False).reset_index(drop=True)

    print(f"✅ Search complete in {time.time() - start_time:.1f}s")
    return ranked_df

## Expanding window

### 1 year

In [20]:
import pandas as pd
import numpy as np
from itertools import combinations

pd.set_option("future.no_silent_downcasting", True)

# =========================================================
# SETTINGS
# =========================================================
HORIZON_Y = 1
DRAWDOWN_THRESH = 0.15
MAX_COMBO_SIZE = 5
N_FINALISTS = 20
ALERT_THRESHOLD = 0.10
WARMUP_DAYS = 252
MIN_CRASHES_FOLD = 2

N_FOLDS = 5
PURGE_DAYS = int(HORIZON_Y * 252)
CORR_THRESHOLD = 0.95

SMOOTH_DAYS = 20
RESET_DAYS = 21

MIN_ROWS_TEST = 63
TOP_N_GROUPS_TO_AUDIT = 5

SAMPLE_START_DATE = "1993-01-01"

DAILY_TOLERANCE_DAYS = 7
SLOW_TOLERANCE_DAYS = 90

SHORT_TERM_TECHNICALS = {
    "MACD",
    "RSI (14)",
    "BB Upper",
    "BB Lower",
    "SPX SMA50",
    "SPX SMA200"
}

BINARY_INDICATORS = ["GDP Recession Indicator", "Sahm Rule"]

# =========================================================
# CRASH REGISTRY
# =========================================================
CRASH_REGISTRY = [
    ("Great Depression",        "1929-09-03", "1932-07-08", "financial_crisis", "Credit collapse"),
    ("1973–74 Crash",           "1973-01-11", "1974-10-03", "supply_shock",     "Oil embargo"),
    ("Volcker Recession",       "1980-11-28", "1982-08-12", "policy_shock",     "Rate tightening"),
    ("Black Monday",            "1987-08-25", "1987-12-04", "liquidity_crisis", "Crash cascade"),
    ("LTCM Crisis",             "1998-07-17", "1998-10-08", "liquidity_crisis", "Leverage unwind"),
    ("Global Financial Crisis", "2007-10-09", "2009-03-09", "financial_crisis", "Banking collapse"),
    ("Euro Debt Crisis",        "2011-04-29", "2011-10-03", "financial_crisis", "Sovereign stress"),
    ("COVID Crash",             "2020-02-19", "2020-03-23", "exogenous_shock",  "Pandemic"),
    ("1962 Crash",              "1961-12-12", "1962-06-26", "valuation_bubble", "Speculative unwind"),
    ("Go-Go Years Bust",        "1968-11-29", "1970-05-26", "valuation_bubble", "Conglomerate bubble"),
    ("Dot-com Bust",            "2000-03-24", "2002-10-09", "valuation_bubble", "Tech collapse"),
    ("1994 Bond Shock",         "1994-02-01", "1994-12-08", "policy_shock",     "Fast tightening"),
    ("2018 Tightening Crash",   "2018-09-20", "2018-12-24", "policy_shock",     "QT + hikes"),
    ("2022 Rate Shock",         "2022-01-03", "2022-10-12", "policy_shock",     "Inflation"),
    ("Post-WWII Recession",     "1946-05-29", "1947-05-19", "demand_shock",     "Demobilisation"),
    ("Korean War Slowdown",     "1953-01-05", "1953-09-14", "demand_shock",     "Slowdown"),
    ("1990 Recession",          "1990-07-16", "1990-10-11", "demand_shock",     "Oil spike"),
    ("2001 Recession",          "2001-05-21", "2001-09-21", "demand_shock",     "Post-bubble"),
    ("1966 Bear Market",        "1966-02-09", "1966-10-07", "soft_bear",        "Growth scare"),
    ("2015–16 China Scare",     "2015-05-21", "2016-02-11", "soft_bear",        "China stress"),
]

CRASH_REGISTRY_DF = pd.DataFrame(
    CRASH_REGISTRY,
    columns=["crash_name", "peak_date", "trough_date", "regime_type", "description"]
)
CRASH_REGISTRY_DF["peak_date"] = pd.to_datetime(CRASH_REGISTRY_DF["peak_date"])
CRASH_REGISTRY_DF["trough_date"] = pd.to_datetime(CRASH_REGISTRY_DF["trough_date"])

# =========================================================
# HELPERS
# =========================================================
def ensure_date_col(df):
    df = df.copy()
    if "Date" not in df.columns:
        df = df.reset_index()
    df["Date"] = pd.to_datetime(df["Date"])
    if getattr(df["Date"].dt, "tz", None) is not None:
        df["Date"] = df["Date"].dt.tz_convert("UTC").dt.tz_localize(None)
    return df.sort_values("Date").reset_index(drop=True)


def get_clean_indicator(name):
    df = ensure_date_col(indicators[name])
    val_col = [c for c in df.columns if c != "Date"][0]
    out = df[["Date", val_col]].rename(columns={val_col: name})
    return out.sort_values("Date").reset_index(drop=True)


def infer_tolerance_days(ind_name, ind_df):
    dates = pd.to_datetime(ind_df["Date"]).sort_values().dropna()
    if len(dates) < 5:
        return SLOW_TOLERANCE_DAYS
    median_gap = dates.diff().dropna().dt.days.median()
    if pd.isna(median_gap):
        return SLOW_TOLERANCE_DAYS
    elif median_gap <= 3:
        return DAILY_TOLERANCE_DAYS
    else:
        return SLOW_TOLERANCE_DAYS


def expanding_percentile_rank(series: pd.Series, min_periods: int = 63) -> pd.Series:
    """
    At each row i, computes the percentile rank of series[i] within
    series[0:i+1]. This is the correct leak-free replacement for the
    rolling trailing_percentile that was previously computed on the
    full dataset before the fold split.

    For binary / near-binary indicators we smooth first (same logic as
    before) so that the expanding rank still produces a meaningful signal.
    """
    arr = series.values.astype(float)
    out = np.full(len(arr), np.nan)

    for i in range(len(arr)):
        window = arr[: i + 1]
        valid = window[~np.isnan(window)]
        if len(valid) < min_periods:
            continue
        current = arr[i]
        if np.isnan(current):
            continue
        # rank of current value within the expanding window
        out[i] = np.mean(valid <= current)

    return pd.Series(out, index=series.index)


def make_signal_expanding(series: pd.Series, warmup: int = 63, name: str = "") -> pd.Series:
    """
    Leak-free signal builder using expanding percentile rank.

    Binary indicators are first smoothed with a rolling mean (same as
    before) so that the raw 0/1 values don't dominate; then the
    expanding rank is applied to the smoothed series.
    """
    s = pd.Series(series, dtype=float)

    if name in BINARY_INDICATORS or s.dropna().nunique() <= 2:
        # smooth first, then rank
        s = s.rolling(window=252, min_periods=warmup).mean()

    return expanding_percentile_rank(s, min_periods=warmup)


def make_target(prices, dates, horizon_days=504, threshold=0.10):
    s = pd.Series(prices).reset_index(drop=True)
    dates_s = pd.Series(pd.to_datetime(dates)).reset_index(drop=True)

    future_min = s[::-1].rolling(horizon_days, min_periods=horizon_days).min()[::-1]
    label = ((future_min / s) - 1 <= -threshold).astype(float)
    label[future_min.isna()] = np.nan

    crash_name_col = pd.Series([""] * len(s), dtype=object)
    crash_start_col = pd.Series(pd.NaT, index=range(len(s)), dtype="datetime64[ns]")

    for _, row in CRASH_REGISTRY_DF.iterrows():
        window_start = row["peak_date"] - pd.Timedelta(days=horizon_days)
        mask = (dates_s >= window_start) & (dates_s <= row["peak_date"])
        crash_name_col.loc[mask] = row["crash_name"]
        crash_start_col.loc[mask] = row["peak_date"]

    return label.values, crash_name_col.values, crash_start_col.values


def compute_regime_score(m):
    lead_bonus = min(2.0, 1 + (m["avg_lead"] / 252))
    return m["recall"] * lead_bonus * m["precision"], {}


# =========================================================
# EVENT ENGINE
# =========================================================
def extract_discrete_events(is_on, reset_days=RESET_DAYS):
    is_on = pd.Series(is_on).fillna(False).astype(bool)
    events = np.zeros(len(is_on), dtype=int)
    last_event_idx = -(10 ** 9)
    prev_on = False

    for i in range(len(is_on)):
        if is_on.iloc[i]:
            if (not prev_on) or ((i - last_event_idx) >= reset_days):
                events[i] = 1
                last_event_idx = i
            prev_on = True
        else:
            prev_on = False

    return events


def build_trigger_dates(signal_series, train_threshold, smooth_days=SMOOTH_DAYS, reset_days=RESET_DAYS):
    s = pd.Series(signal_series, dtype=float)
    smoothed = s.rolling(smooth_days, min_periods=smooth_days).mean()
    is_on = smoothed >= train_threshold
    events = extract_discrete_events(is_on, reset_days=reset_days)

    return pd.DataFrame({
        "signal": s,
        "smoothed_signal": smoothed,
        "is_on": is_on,
        "event": events
    })


def compute_event_metrics(event_starts, trigger_dates, lookback_days=504):
    event_starts = pd.to_datetime(pd.Series(event_starts)).dropna().sort_values().reset_index(drop=True)
    trigger_dates = pd.to_datetime(pd.Series(trigger_dates)).dropna().sort_values().reset_index(drop=True)

    if len(event_starts) == 0:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "avg_lead": 0.0,
            "crash_detail": [],
            "missed_crashes": []
        }

    used_trigger_idx = set()
    detail = []
    tp = 0
    lead_sum = 0.0
    missed = []

    for ev in event_starts:
        window_start = ev - pd.Timedelta(days=lookback_days)
        eligible = [
            (idx, dt) for idx, dt in enumerate(trigger_dates)
            if idx not in used_trigger_idx and window_start <= dt < ev
        ]

        if len(eligible) == 0:
            detail.append({
                "crash": str(ev.date()),
                "caught": False,
                "lead_days": 0,
                "first_trigger": pd.NaT
            })
            missed.append(ev)
            continue

        idx, first = eligible[0]
        used_trigger_idx.add(idx)
        tp += 1
        lead_days = (ev - first).days
        lead_sum += lead_days
        detail.append({
            "crash": str(ev.date()),
            "caught": True,
            "lead_days": lead_days,
            "first_trigger": first
        })

    precision = tp / len(trigger_dates) if len(trigger_dates) else 0.0
    recall = tp / len(event_starts) if len(event_starts) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    avg_lead = lead_sum / tp if tp else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "avg_lead": avg_lead,
        "crash_detail": detail,
        "missed_crashes": missed
    }


def fit_train_threshold(signal_series, alert_threshold=ALERT_THRESHOLD, smooth_days=SMOOTH_DAYS):
    s = pd.Series(signal_series, dtype=float)
    smoothed = s.rolling(smooth_days, min_periods=smooth_days).mean().dropna()
    if len(smoothed) == 0:
        return np.nan
    return np.nanpercentile(smoothed, 100 * (1 - alert_threshold))


# =========================================================
# CORRELATION PRUNING
# Note: now called with fold-specific train_df, so correlation
# is computed only on training data — no lookahead.
# =========================================================
def filter_correlated_finalists_late(df, ranked_names, threshold=CORR_THRESHOLD):
    cols = [f"{n}__pct" for n in ranked_names if f"{n}__pct" in df.columns]
    if len(cols) <= 1:
        return ranked_names

    corr = df[cols].corr().abs()
    kept = []

    for name in ranked_names:
        col = f"{name}__pct"
        if col not in corr.columns:
            continue

        should_keep = True
        replace_idx = None

        for j, kept_name in enumerate(kept):
            kept_col = f"{kept_name}__pct"
            if kept_col not in corr.columns:
                continue

            if corr.loc[col, kept_col] > threshold:
                if (name not in SHORT_TERM_TECHNICALS) and (kept_name in SHORT_TERM_TECHNICALS):
                    replace_idx = j
                    should_keep = True
                else:
                    should_keep = False
                break

        if replace_idx is not None:
            kept[replace_idx] = name
        elif should_keep:
            kept.append(name)

    return kept


# =========================================================
# DATASET
# Build master_ds WITHOUT percentile columns.
# Percentile signals are computed inside each fold using only
# data up to (and including) that row — no future peeking.
# =========================================================
def build_master_ds():
    spx = ensure_date_col(spx_c)
    price_col = [c for c in spx.columns if c != "Date"][0]
    spx = spx[["Date", price_col]].rename(columns={price_col: "close"}).sort_values("Date").reset_index(drop=True)

    spx["target_downturn"], spx["crash_name"], spx["crash_start"] = make_target(
        spx["close"].values,
        spx["Date"].values,
        int(HORIZON_Y * 252),
        DRAWDOWN_THRESH
    )

    master_ds = spx[["Date", "target_downturn", "crash_name", "crash_start", "close"]].copy()

    for name in sorted(indicators.keys()):
        try:
            ind = get_clean_indicator(name)
            tol_days = infer_tolerance_days(name, ind)

            master_ds = pd.merge_asof(
                master_ds.sort_values("Date"),
                ind.sort_values("Date"),
                on="Date",
                tolerance=pd.Timedelta(days=tol_days),
                direction="backward"
            )
            # Forward-fill raw indicator values only — no percentile here
            master_ds[name] = master_ds[name].ffill()

        except Exception as e:
            print(f"Skipping {name}: {e}")

    master_ds = master_ds.loc[master_ds["Date"] >= pd.Timestamp(SAMPLE_START_DATE)].copy()
    return master_ds.reset_index(drop=True)


def add_expanding_percentile_signals(df: pd.DataFrame, indicator_names: list) -> pd.DataFrame:
    """
    Compute leak-free expanding-window percentile ranks for all
    indicators on the given dataframe slice.

    Key point: this must always be called on a contiguous prefix of the
    full dataset (i.e. all rows from the start up to the fold boundary),
    NOT on a slice starting mid-series.  The fold loop passes
    master_ds.iloc[:test_end] for exactly this reason.
    """
    df = df.copy()
    for name in indicator_names:
        if name not in df.columns:
            continue
        df[f"{name}__pct"] = make_signal_expanding(df[name], warmup=WARMUP_DAYS, name=name)
    return df


# =========================================================
# MAIN SEARCH
# =========================================================
def run_macro_search_with_event_reset():
    # master_ds holds raw indicator values only — no percentile columns yet
    master_ds = build_master_ds().dropna(subset=["target_downturn"]).reset_index(drop=True)
    all_names = sorted(indicators.keys())

    fold_size = (len(master_ds) - WARMUP_DAYS) // N_FOLDS
    fold_combo_scores = []

    for i in range(N_FOLDS):
        test_start = WARMUP_DAYS + i * fold_size
        test_end = WARMUP_DAYS + (i + 1) * fold_size if i < N_FOLDS - 1 else len(master_ds)

        # Purge gap: training ends well before the test window to avoid
        # horizon overlap (unchanged from original)
        train_end = max(WARMUP_DAYS, test_start - PURGE_DAYS)

        # ── LEAK-FREE PERCENTILE COMPUTATION ──────────────────────────
        # We pass the full prefix up to test_end so that the expanding
        # window at row (test_end - 1) has seen all rows before it.
        # Training rows see only rows before them; test rows see only
        # rows before them — no future data ever enters the rank.
        prefix_with_pct = add_expanding_percentile_signals(
            master_ds.iloc[:test_end].copy(),
            all_names
        )

        train_df = prefix_with_pct.iloc[WARMUP_DAYS:train_end].copy()
        test_df  = prefix_with_pct.iloc[test_start:test_end].copy()
        # ──────────────────────────────────────────────────────────────

        if len(train_df) < 504 or len(test_df) < MIN_ROWS_TEST:
            continue

        event_starts_train = train_df.loc[train_df["crash_start"].notna(), "crash_start"].drop_duplicates()
        event_starts_test  = test_df.loc[test_df["crash_start"].notna(), "crash_start"].drop_duplicates()

        if len(event_starts_train) < MIN_CRASHES_FOLD or len(event_starts_test) < 1:
            continue

        # ── FINALIST RANKING (on train only) ──────────────────────────
        finalist_scores = []
        for n in all_names:
            col = f"{n}__pct"
            if col not in train_df.columns:
                continue

            sub_train = train_df[["Date", "crash_start", col]].dropna().copy()
            if len(sub_train) < 252:
                continue

            threshold = fit_train_threshold(sub_train[col])
            if pd.isna(threshold):
                continue

            trig_engine = build_trigger_dates(sub_train[col], threshold)
            trigger_dates_train = sub_train.loc[trig_engine["event"] == 1, "Date"]

            m = compute_event_metrics(
                event_starts_train,
                trigger_dates_train,
                lookback_days=int(HORIZON_Y * 252)
            )

            finalist_scores.append({
                "name": n,
                "train_f1": m["f1"],
                "train_recall": m["recall"],
                "train_precision": m["precision"]
            })

        if len(finalist_scores) == 0:
            continue

        ranked_finalists = (
            pd.DataFrame(finalist_scores)
            .sort_values(["train_f1", "train_recall"], ascending=False)
            .head(N_FINALISTS * 2)
            ["name"]
            .tolist()
        )

        # Correlation pruning uses train_df only — also leak-free
        finalists = filter_correlated_finalists_late(train_df, ranked_finalists)
        finalists = finalists[:N_FINALISTS]

        # ── COMBO EVALUATION ──────────────────────────────────────────
        for k in range(1, min(MAX_COMBO_SIZE, len(finalists)) + 1):
            for combo in combinations(finalists, k):
                cols = [f"{c}__pct" for c in combo]

                sub_train = train_df[["Date", "crash_start"] + cols].dropna().copy()
                sub_test  = test_df[["Date", "crash_start"] + cols].dropna().copy()

                if len(sub_train) < 252 or len(sub_test) < MIN_ROWS_TEST:
                    continue

                sig_train = sub_train[cols].mean(axis=1)
                sig_test  = sub_test[cols].mean(axis=1)

                threshold = fit_train_threshold(sig_train)
                if pd.isna(threshold):
                    continue

                trig_test = build_trigger_dates(sig_test, threshold)
                trigger_dates_test = sub_test.loc[trig_test["event"] == 1, "Date"]

                m = compute_event_metrics(
                    event_starts_test,
                    trigger_dates_test,
                    lookback_days=int(HORIZON_Y * 252)
                )
                score, _ = compute_regime_score(m)

                fold_combo_scores.append({
                    "combo": " + ".join(combo),
                    "fold": i + 1,
                    "size": k,
                    "f1": m["f1"],
                    "recall": m["recall"],
                    "prec": m["precision"],
                    "avg_lead": m["avg_lead"],
                    "score": score
                })

    if not fold_combo_scores:
        return pd.DataFrame(), pd.DataFrame()

    fold_results = pd.DataFrame(fold_combo_scores)

    agg = (
        fold_results
        .groupby("combo")
        .agg(
            mean_f1=("f1", "mean"),
            mean_rec=("recall", "mean"),
            mean_prec=("prec", "mean"),
            mean_lead=("avg_lead", "mean"),
            mean_score=("score", "mean"),
            n_folds=("fold", "nunique"),
            size=("size", "first")
        )
        .reset_index()
        .query("n_folds >= 3")
        .sort_values(["n_folds", "mean_score"], ascending=[False, False])
        .reset_index(drop=True)
    )

    return agg, fold_results


# =========================================================
# RUN
# =========================================================
agg_results, fold_results = run_macro_search_with_event_reset()

print("\n=== TOP GROUPS ===")
display(
    agg_results.head(20).style.background_gradient(
        subset=["mean_score", "mean_rec", "mean_prec"],
        cmap="RdYlGn"
    )
)

print("\n=== FOLD-LEVEL RESULTS ===")
display(
    fold_results.head(50).style.background_gradient(
        subset=["score", "recall", "prec"],
        cmap="RdYlGn"
    )
)


=== TOP GROUPS ===


,combo,mean_f1,mean_rec,mean_prec,mean_lead,mean_score,n_folds,size
0,MACD + RSI (14),0.746032,1.000000,0.633333,175.666667,1.094378,3,2
1,RSI (14),0.688889,1.000000,0.583333,180.333333,1.014881,3,1
2,AAII Bull-Bear Spread + SPX SMA50 + RSI (14) + Brent Oil,0.722222,0.666667,0.833333,128.166667,0.936177,3,4
3,AAII Bull-Bear Spread + SPX SMA50 + RSI (14),0.755556,0.833333,0.777778,129.166667,0.929674,3,3
4,AAII Bull-Bear Spread + RSI (14),0.688889,0.666667,0.777778,146.000000,0.905423,3,2
5,MACD + AAII Bullish + SPX SMA200,0.500000,0.500000,0.500000,147.333333,0.766204,3,3
6,SPX SMA50 + RSI (14) + SPX SMA200,0.603175,0.833333,0.600000,172.500000,0.728571,3,3
7,AAII Bull-Bear Spread + SPX SMA50 + RSI (14) + SPX SMA200,0.533333,0.666667,0.444444,94.666667,0.694885,3,4
8,AAII Bull-Bear Spread + RSI (14) + SPX SMA200,0.533333,0.666667,0.444444,94.333333,0.694004,3,3
9,SPX SMA50 + RSI (14) + Brent Oil,0.600000,0.833333,0.622222,144.166667,0.689903,3,3



=== FOLD-LEVEL RESULTS ===


,combo,fold,size,f1,recall,prec,avg_lead,score
0,AAII Bull-Bear Spread,3,1,0.000000,0.000000,0.000000,0.000000,0.000000
1,CBOE SKEW,3,1,0.333333,1.000000,0.200000,159.000000,0.326190
2,Gold / Brent Ratio,3,1,0.000000,0.000000,0.000000,0.000000,0.000000
3,SPX SMA50,3,1,0.666667,0.500000,1.000000,99.000000,0.696429
4,BB Lower,3,1,0.500000,0.500000,0.500000,132.000000,0.380952
5,BBB–Treasury Spread,3,1,0.000000,0.000000,0.000000,0.000000,0.000000
6,Corp Profit Margin,3,1,0.222222,1.000000,0.125000,229.000000,0.238591
7,Gold,3,1,0.222222,1.000000,0.125000,229.000000,0.238591
8,MACD,3,1,0.000000,0.000000,0.000000,0.000000,0.000000
9,Sahm Rule,3,1,0.285714,0.500000,0.200000,234.000000,0.192857


### 5 years

In [19]:
import pandas as pd
import numpy as np
from itertools import combinations

pd.set_option("future.no_silent_downcasting", True)

# =========================================================
# SETTINGS
# =========================================================
HORIZON_Y = 5
DRAWDOWN_THRESH = 0.15
MAX_COMBO_SIZE = 4
N_FINALISTS = 20
ALERT_THRESHOLD = 0.10
WARMUP_DAYS = 252
MIN_CRASHES_FOLD = 2

N_FOLDS = 5
PURGE_DAYS = int(HORIZON_Y * 252)
CORR_THRESHOLD = 0.95

SMOOTH_DAYS = 20
RESET_DAYS = 21

MIN_ROWS_TEST = 63
TOP_N_GROUPS_TO_AUDIT = 5

SAMPLE_START_DATE = "1993-01-01"

DAILY_TOLERANCE_DAYS = 7
SLOW_TOLERANCE_DAYS = 90

SHORT_TERM_TECHNICALS = {
    "MACD",
    "RSI (14)",
    "BB Upper",
    "BB Lower",
    "SPX SMA50",
    "SPX SMA200"
}

BINARY_INDICATORS = ["GDP Recession Indicator", "Sahm Rule"]

# =========================================================
# CRASH REGISTRY
# =========================================================
CRASH_REGISTRY = [
    ("Great Depression",        "1929-09-03", "1932-07-08", "financial_crisis", "Credit collapse"),
    ("1973–74 Crash",           "1973-01-11", "1974-10-03", "supply_shock",     "Oil embargo"),
    ("Volcker Recession",       "1980-11-28", "1982-08-12", "policy_shock",     "Rate tightening"),
    ("Black Monday",            "1987-08-25", "1987-12-04", "liquidity_crisis", "Crash cascade"),
    ("LTCM Crisis",             "1998-07-17", "1998-10-08", "liquidity_crisis", "Leverage unwind"),
    ("Global Financial Crisis", "2007-10-09", "2009-03-09", "financial_crisis", "Banking collapse"),
    ("Euro Debt Crisis",        "2011-04-29", "2011-10-03", "financial_crisis", "Sovereign stress"),
    ("COVID Crash",             "2020-02-19", "2020-03-23", "exogenous_shock",  "Pandemic"),
    ("1962 Crash",              "1961-12-12", "1962-06-26", "valuation_bubble", "Speculative unwind"),
    ("Go-Go Years Bust",        "1968-11-29", "1970-05-26", "valuation_bubble", "Conglomerate bubble"),
    ("Dot-com Bust",            "2000-03-24", "2002-10-09", "valuation_bubble", "Tech collapse"),
    ("1994 Bond Shock",         "1994-02-01", "1994-12-08", "policy_shock",     "Fast tightening"),
    ("2018 Tightening Crash",   "2018-09-20", "2018-12-24", "policy_shock",     "QT + hikes"),
    ("2022 Rate Shock",         "2022-01-03", "2022-10-12", "policy_shock",     "Inflation"),
    ("Post-WWII Recession",     "1946-05-29", "1947-05-19", "demand_shock",     "Demobilisation"),
    ("Korean War Slowdown",     "1953-01-05", "1953-09-14", "demand_shock",     "Slowdown"),
    ("1990 Recession",          "1990-07-16", "1990-10-11", "demand_shock",     "Oil spike"),
    ("2001 Recession",          "2001-05-21", "2001-09-21", "demand_shock",     "Post-bubble"),
    ("1966 Bear Market",        "1966-02-09", "1966-10-07", "soft_bear",        "Growth scare"),
    ("2015–16 China Scare",     "2015-05-21", "2016-02-11", "soft_bear",        "China stress"),
]

CRASH_REGISTRY_DF = pd.DataFrame(
    CRASH_REGISTRY,
    columns=["crash_name", "peak_date", "trough_date", "regime_type", "description"]
)
CRASH_REGISTRY_DF["peak_date"] = pd.to_datetime(CRASH_REGISTRY_DF["peak_date"])
CRASH_REGISTRY_DF["trough_date"] = pd.to_datetime(CRASH_REGISTRY_DF["trough_date"])

# =========================================================
# HELPERS
# =========================================================
def ensure_date_col(df):
    df = df.copy()
    if "Date" not in df.columns:
        df = df.reset_index()
    df["Date"] = pd.to_datetime(df["Date"])
    if getattr(df["Date"].dt, "tz", None) is not None:
        df["Date"] = df["Date"].dt.tz_convert("UTC").dt.tz_localize(None)
    return df.sort_values("Date").reset_index(drop=True)


def get_clean_indicator(name):
    df = ensure_date_col(indicators[name])
    val_col = [c for c in df.columns if c != "Date"][0]
    out = df[["Date", val_col]].rename(columns={val_col: name})
    return out.sort_values("Date").reset_index(drop=True)


def infer_tolerance_days(ind_name, ind_df):
    dates = pd.to_datetime(ind_df["Date"]).sort_values().dropna()
    if len(dates) < 5:
        return SLOW_TOLERANCE_DAYS
    median_gap = dates.diff().dropna().dt.days.median()
    if pd.isna(median_gap):
        return SLOW_TOLERANCE_DAYS
    elif median_gap <= 3:
        return DAILY_TOLERANCE_DAYS
    else:
        return SLOW_TOLERANCE_DAYS


def expanding_percentile_rank(series: pd.Series, min_periods: int = 63) -> pd.Series:
    """
    At each row i, computes the percentile rank of series[i] within
    series[0:i+1]. This is the correct leak-free replacement for the
    rolling trailing_percentile that was previously computed on the
    full dataset before the fold split.

    For binary / near-binary indicators we smooth first (same logic as
    before) so that the expanding rank still produces a meaningful signal.
    """
    arr = series.values.astype(float)
    out = np.full(len(arr), np.nan)

    for i in range(len(arr)):
        window = arr[: i + 1]
        valid = window[~np.isnan(window)]
        if len(valid) < min_periods:
            continue
        current = arr[i]
        if np.isnan(current):
            continue
        # rank of current value within the expanding window
        out[i] = np.mean(valid <= current)

    return pd.Series(out, index=series.index)


def make_signal_expanding(series: pd.Series, warmup: int = 63, name: str = "") -> pd.Series:
    """
    Leak-free signal builder using expanding percentile rank.

    Binary indicators are first smoothed with a rolling mean (same as
    before) so that the raw 0/1 values don't dominate; then the
    expanding rank is applied to the smoothed series.
    """
    s = pd.Series(series, dtype=float)

    if name in BINARY_INDICATORS or s.dropna().nunique() <= 2:
        # smooth first, then rank
        s = s.rolling(window=252, min_periods=warmup).mean()

    return expanding_percentile_rank(s, min_periods=warmup)


def make_target(prices, dates, horizon_days=504, threshold=0.10):
    s = pd.Series(prices).reset_index(drop=True)
    dates_s = pd.Series(pd.to_datetime(dates)).reset_index(drop=True)

    future_min = s[::-1].rolling(horizon_days, min_periods=horizon_days).min()[::-1]
    label = ((future_min / s) - 1 <= -threshold).astype(float)
    label[future_min.isna()] = np.nan

    crash_name_col = pd.Series([""] * len(s), dtype=object)
    crash_start_col = pd.Series(pd.NaT, index=range(len(s)), dtype="datetime64[ns]")

    for _, row in CRASH_REGISTRY_DF.iterrows():
        window_start = row["peak_date"] - pd.Timedelta(days=horizon_days)
        mask = (dates_s >= window_start) & (dates_s <= row["peak_date"])
        crash_name_col.loc[mask] = row["crash_name"]
        crash_start_col.loc[mask] = row["peak_date"]

    return label.values, crash_name_col.values, crash_start_col.values


def compute_regime_score(m):
    lead_bonus = min(2.0, 1 + (m["avg_lead"] / 252))
    return m["recall"] * lead_bonus * m["precision"], {}


# =========================================================
# EVENT ENGINE
# =========================================================
def extract_discrete_events(is_on, reset_days=RESET_DAYS):
    is_on = pd.Series(is_on).fillna(False).astype(bool)
    events = np.zeros(len(is_on), dtype=int)
    last_event_idx = -(10 ** 9)
    prev_on = False

    for i in range(len(is_on)):
        if is_on.iloc[i]:
            if (not prev_on) or ((i - last_event_idx) >= reset_days):
                events[i] = 1
                last_event_idx = i
            prev_on = True
        else:
            prev_on = False

    return events


def build_trigger_dates(signal_series, train_threshold, smooth_days=SMOOTH_DAYS, reset_days=RESET_DAYS):
    s = pd.Series(signal_series, dtype=float)
    smoothed = s.rolling(smooth_days, min_periods=smooth_days).mean()
    is_on = smoothed >= train_threshold
    events = extract_discrete_events(is_on, reset_days=reset_days)

    return pd.DataFrame({
        "signal": s,
        "smoothed_signal": smoothed,
        "is_on": is_on,
        "event": events
    })


def compute_event_metrics(event_starts, trigger_dates, lookback_days=504):
    event_starts = pd.to_datetime(pd.Series(event_starts)).dropna().sort_values().reset_index(drop=True)
    trigger_dates = pd.to_datetime(pd.Series(trigger_dates)).dropna().sort_values().reset_index(drop=True)

    if len(event_starts) == 0:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "avg_lead": 0.0,
            "crash_detail": [],
            "missed_crashes": []
        }

    used_trigger_idx = set()
    detail = []
    tp = 0
    lead_sum = 0.0
    missed = []

    for ev in event_starts:
        window_start = ev - pd.Timedelta(days=lookback_days)
        eligible = [
            (idx, dt) for idx, dt in enumerate(trigger_dates)
            if idx not in used_trigger_idx and window_start <= dt < ev
        ]

        if len(eligible) == 0:
            detail.append({
                "crash": str(ev.date()),
                "caught": False,
                "lead_days": 0,
                "first_trigger": pd.NaT
            })
            missed.append(ev)
            continue

        idx, first = eligible[0]
        used_trigger_idx.add(idx)
        tp += 1
        lead_days = (ev - first).days
        lead_sum += lead_days
        detail.append({
            "crash": str(ev.date()),
            "caught": True,
            "lead_days": lead_days,
            "first_trigger": first
        })

    precision = tp / len(trigger_dates) if len(trigger_dates) else 0.0
    recall = tp / len(event_starts) if len(event_starts) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    avg_lead = lead_sum / tp if tp else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "avg_lead": avg_lead,
        "crash_detail": detail,
        "missed_crashes": missed
    }


def fit_train_threshold(signal_series, alert_threshold=ALERT_THRESHOLD, smooth_days=SMOOTH_DAYS):
    s = pd.Series(signal_series, dtype=float)
    smoothed = s.rolling(smooth_days, min_periods=smooth_days).mean().dropna()
    if len(smoothed) == 0:
        return np.nan
    return np.nanpercentile(smoothed, 100 * (1 - alert_threshold))


# =========================================================
# CORRELATION PRUNING
# Note: now called with fold-specific train_df, so correlation
# is computed only on training data — no lookahead.
# =========================================================
def filter_correlated_finalists_late(df, ranked_names, threshold=CORR_THRESHOLD):
    cols = [f"{n}__pct" for n in ranked_names if f"{n}__pct" in df.columns]
    if len(cols) <= 1:
        return ranked_names

    corr = df[cols].corr().abs()
    kept = []

    for name in ranked_names:
        col = f"{name}__pct"
        if col not in corr.columns:
            continue

        should_keep = True
        replace_idx = None

        for j, kept_name in enumerate(kept):
            kept_col = f"{kept_name}__pct"
            if kept_col not in corr.columns:
                continue

            if corr.loc[col, kept_col] > threshold:
                if (name not in SHORT_TERM_TECHNICALS) and (kept_name in SHORT_TERM_TECHNICALS):
                    replace_idx = j
                    should_keep = True
                else:
                    should_keep = False
                break

        if replace_idx is not None:
            kept[replace_idx] = name
        elif should_keep:
            kept.append(name)

    return kept


# =========================================================
# DATASET
# Build master_ds WITHOUT percentile columns.
# Percentile signals are computed inside each fold using only
# data up to (and including) that row — no future peeking.
# =========================================================
def build_master_ds():
    spx = ensure_date_col(spx_c)
    price_col = [c for c in spx.columns if c != "Date"][0]
    spx = spx[["Date", price_col]].rename(columns={price_col: "close"}).sort_values("Date").reset_index(drop=True)

    spx["target_downturn"], spx["crash_name"], spx["crash_start"] = make_target(
        spx["close"].values,
        spx["Date"].values,
        int(HORIZON_Y * 252),
        DRAWDOWN_THRESH
    )

    master_ds = spx[["Date", "target_downturn", "crash_name", "crash_start", "close"]].copy()

    for name in sorted(indicators.keys()):
        try:
            ind = get_clean_indicator(name)
            tol_days = infer_tolerance_days(name, ind)

            master_ds = pd.merge_asof(
                master_ds.sort_values("Date"),
                ind.sort_values("Date"),
                on="Date",
                tolerance=pd.Timedelta(days=tol_days),
                direction="backward"
            )
            # Forward-fill raw indicator values only — no percentile here
            master_ds[name] = master_ds[name].ffill()

        except Exception as e:
            print(f"Skipping {name}: {e}")

    master_ds = master_ds.loc[master_ds["Date"] >= pd.Timestamp(SAMPLE_START_DATE)].copy()
    return master_ds.reset_index(drop=True)


def add_expanding_percentile_signals(df: pd.DataFrame, indicator_names: list) -> pd.DataFrame:
    """
    Compute leak-free expanding-window percentile ranks for all
    indicators on the given dataframe slice.

    Key point: this must always be called on a contiguous prefix of the
    full dataset (i.e. all rows from the start up to the fold boundary),
    NOT on a slice starting mid-series.  The fold loop passes
    master_ds.iloc[:test_end] for exactly this reason.
    """
    df = df.copy()
    for name in indicator_names:
        if name not in df.columns:
            continue
        df[f"{name}__pct"] = make_signal_expanding(df[name], warmup=WARMUP_DAYS, name=name)
    return df


# =========================================================
# MAIN SEARCH
# =========================================================
def run_macro_search_with_event_reset():
    # master_ds holds raw indicator values only — no percentile columns yet
    master_ds = build_master_ds().dropna(subset=["target_downturn"]).reset_index(drop=True)
    all_names = sorted(indicators.keys())

    fold_size = (len(master_ds) - WARMUP_DAYS) // N_FOLDS
    fold_combo_scores = []

    for i in range(N_FOLDS):
        test_start = WARMUP_DAYS + i * fold_size
        test_end = WARMUP_DAYS + (i + 1) * fold_size if i < N_FOLDS - 1 else len(master_ds)

        # Purge gap: training ends well before the test window to avoid
        # horizon overlap (unchanged from original)
        train_end = max(WARMUP_DAYS, test_start - PURGE_DAYS)

        # ── LEAK-FREE PERCENTILE COMPUTATION ──────────────────────────
        # We pass the full prefix up to test_end so that the expanding
        # window at row (test_end - 1) has seen all rows before it.
        # Training rows see only rows before them; test rows see only
        # rows before them — no future data ever enters the rank.
        prefix_with_pct = add_expanding_percentile_signals(
            master_ds.iloc[:test_end].copy(),
            all_names
        )

        train_df = prefix_with_pct.iloc[WARMUP_DAYS:train_end].copy()
        test_df  = prefix_with_pct.iloc[test_start:test_end].copy()
        # ──────────────────────────────────────────────────────────────

        if len(train_df) < 504 or len(test_df) < MIN_ROWS_TEST:
            continue

        event_starts_train = train_df.loc[train_df["crash_start"].notna(), "crash_start"].drop_duplicates()
        event_starts_test  = test_df.loc[test_df["crash_start"].notna(), "crash_start"].drop_duplicates()

        if len(event_starts_train) < MIN_CRASHES_FOLD or len(event_starts_test) < 1:
            continue

        # ── FINALIST RANKING (on train only) ──────────────────────────
        finalist_scores = []
        for n in all_names:
            col = f"{n}__pct"
            if col not in train_df.columns:
                continue

            sub_train = train_df[["Date", "crash_start", col]].dropna().copy()
            if len(sub_train) < 252:
                continue

            threshold = fit_train_threshold(sub_train[col])
            if pd.isna(threshold):
                continue

            trig_engine = build_trigger_dates(sub_train[col], threshold)
            trigger_dates_train = sub_train.loc[trig_engine["event"] == 1, "Date"]

            m = compute_event_metrics(
                event_starts_train,
                trigger_dates_train,
                lookback_days=int(HORIZON_Y * 252)
            )

            finalist_scores.append({
                "name": n,
                "train_f1": m["f1"],
                "train_recall": m["recall"],
                "train_precision": m["precision"]
            })

        if len(finalist_scores) == 0:
            continue

        ranked_finalists = (
            pd.DataFrame(finalist_scores)
            .sort_values(["train_f1", "train_recall"], ascending=False)
            .head(N_FINALISTS * 2)
            ["name"]
            .tolist()
        )

        # Correlation pruning uses train_df only — also leak-free
        finalists = filter_correlated_finalists_late(train_df, ranked_finalists)
        finalists = finalists[:N_FINALISTS]

        # ── COMBO EVALUATION ──────────────────────────────────────────
        for k in range(1, min(MAX_COMBO_SIZE, len(finalists)) + 1):
            for combo in combinations(finalists, k):
                cols = [f"{c}__pct" for c in combo]

                sub_train = train_df[["Date", "crash_start"] + cols].dropna().copy()
                sub_test  = test_df[["Date", "crash_start"] + cols].dropna().copy()

                if len(sub_train) < 252 or len(sub_test) < MIN_ROWS_TEST:
                    continue

                sig_train = sub_train[cols].mean(axis=1)
                sig_test  = sub_test[cols].mean(axis=1)

                threshold = fit_train_threshold(sig_train)
                if pd.isna(threshold):
                    continue

                trig_test = build_trigger_dates(sig_test, threshold)
                trigger_dates_test = sub_test.loc[trig_test["event"] == 1, "Date"]

                m = compute_event_metrics(
                    event_starts_test,
                    trigger_dates_test,
                    lookback_days=int(HORIZON_Y * 252)
                )
                score, _ = compute_regime_score(m)

                fold_combo_scores.append({
                    "combo": " + ".join(combo),
                    "fold": i + 1,
                    "size": k,
                    "f1": m["f1"],
                    "recall": m["recall"],
                    "prec": m["precision"],
                    "avg_lead": m["avg_lead"],
                    "score": score
                })

    if not fold_combo_scores:
        return pd.DataFrame(), pd.DataFrame()

    fold_results = pd.DataFrame(fold_combo_scores)

    agg = (
        fold_results
        .groupby("combo")
        .agg(
            mean_f1=("f1", "mean"),
            mean_rec=("recall", "mean"),
            mean_prec=("prec", "mean"),
            mean_lead=("avg_lead", "mean"),
            mean_score=("score", "mean"),
            n_folds=("fold", "nunique"),
            size=("size", "first")
        )
        .reset_index()
        .query("n_folds >= 3")
        .sort_values(["n_folds", "mean_score"], ascending=[False, False])
        .reset_index(drop=True)
    )

    return agg, fold_results


# =========================================================
# RUN
# =========================================================
agg_results, fold_results = run_macro_search_with_event_reset()

print("\n=== TOP GROUPS ===")
display(
    agg_results.head(20).style.background_gradient(
        subset=["mean_score", "mean_rec", "mean_prec"],
        cmap="RdYlGn"
    )
)

print("\n=== FOLD-LEVEL RESULTS ===")
display(
    fold_results.head(50).style.background_gradient(
        subset=["score", "recall", "prec"],
        cmap="RdYlGn"
    )
)


=== TOP GROUPS ===


,combo,mean_f1,mean_rec,mean_prec,mean_lead,mean_score,n_folds,size
0,BBB–Treasury Spread + RSI (14),0.688889,0.611111,0.833333,870.166667,1.111111,3,2
1,BBB–Treasury Spread + RSI (14) + MACD,0.577778,0.611111,0.666667,887.666667,0.777778,3,3
2,Equity Risk Premium + AAII Bullish,0.544444,0.555556,0.583333,680.000000,0.694444,3,2
3,RSI (14) + AAII Bull-Bear Spread,0.510101,0.722222,0.527778,684.833333,0.666667,3,2
4,Gold / Brent Ratio + VIX,0.474074,0.722222,0.486111,872.000000,0.638889,3,2
5,Equity Risk Premium + RSI (14) + MACD,0.451852,0.888889,0.337963,758.333333,0.626543,3,3
6,Equity Risk Premium + MACD,0.466667,0.611111,0.566667,662.666667,0.577778,3,2
7,BBB–Treasury Spread + AAII Bearish,0.419192,0.444444,0.683333,981.666667,0.572222,3,2
8,Gold / Brent Ratio + RSI (14),0.388889,0.833333,0.436588,816.055556,0.539843,3,2
9,CBOE SKEW + RSI (14) + AAII Bull-Bear Spread,0.369383,0.888889,0.278986,806.500000,0.539452,3,3



=== FOLD-LEVEL RESULTS ===


,combo,fold,size,f1,recall,prec,avg_lead,score
0,BBB–Treasury Spread,3,1,0.333333,0.500000,0.250000,942.000000,0.250000
1,CBOE SKEW,3,1,0.210526,1.000000,0.117647,884.500000,0.235294
2,Equity Risk Premium,3,1,0.666667,0.500000,1.000000,456.000000,1.000000
3,Gold / Brent Ratio,3,1,0.000000,0.000000,0.000000,0.000000,0.000000
4,Dividend Yield,3,1,0.061538,1.000000,0.031746,1147.500000,0.063492
5,PE Ratio,3,1,0.285714,0.500000,0.200000,821.000000,0.200000
6,AAII Bullish,3,1,0.000000,0.000000,0.000000,0.000000,0.000000
7,VIX,3,1,0.250000,0.500000,0.166667,931.000000,0.166667
8,Yield Curve (10Y–2Y),3,1,0.102564,1.000000,0.054054,1139.000000,0.108108
9,AAII Bearish,3,1,0.071429,1.000000,0.037037,1109.000000,0.074074


# decision tree model

## updated version

In [18]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import TimeSeriesSplit

pd.set_option("future.no_silent_downcasting", True)

# =========================================================
# SETTINGS
# =========================================================
DRAWDOWN_THRESH = 0.15
WARMUP_DAYS = 252

MODEL_TYPE = "random_forest"
MODEL_START_DATE = "1993-01-01"

N_FOLDS = 4
LEAD_DAYS = 252                    # predict crisis start within next 252 trading days
PURGE_DAYS = LEAD_DAYS             # purge equal to forward-looking target window
MIN_TRAIN_ROWS = 756
MIN_TEST_ROWS = 252

SMOOTH_DAYS = 20
ALERT_THRESHOLD = 0.10             # top 10% of smoothed train OOF probs become alarms
RESET_DAYS = 21

MAX_MISSING_FEATURE_FRAC = 0.35
RANDOM_STATE = 42

TREE_MAX_DEPTH = 4
TREE_MIN_SAMPLES_LEAF = 15

RF_N_ESTIMATORS = 300
RF_MAX_DEPTH = 5
RF_MIN_SAMPLES_LEAF = 10

N_FINALISTS = 12
MAX_COMBO_SIZE = 3
MIN_FOLDS_REQUIRED = 2

INNER_OOF_SPLITS = 3               # inner CV for OOF threshold calibration

BINARY_INDICATORS = ["GDP Recession Indicator", "Sahm Rule"]

REGIME_AUGMENT_CANDIDATES = [
    "Shiller CAPE",
    "Tobin Q",
    "PB Ratio",
    "PE Ratio",
    "PS Ratio",
    "Buffet Indicator",
    "Earnings Yield",
    "Dividend Yield",
    "Equity Valuation (OFR)"
]

# =========================================================
# HELPERS
# =========================================================
def ensure_date_col(df):
    df = df.copy()
    if "Date" not in df.columns:
        df = df.reset_index()
    df["Date"] = pd.to_datetime(df["Date"])
    if getattr(df["Date"].dt, "tz", None) is not None:
        df["Date"] = df["Date"].dt.tz_convert("UTC").dt.tz_localize(None)
    return df.sort_values("Date").reset_index(drop=True)


def get_clean_indicator(name):
    df = ensure_date_col(indicators[name])
    val_col = [c for c in df.columns if c != "Date"][0]
    return df[["Date", val_col]].rename(columns={val_col: name}).sort_values("Date").reset_index(drop=True)


def trailing_percentile(x, window=252, min_periods=63):
    s = pd.Series(x)
    return s.rolling(window=window, min_periods=min_periods).apply(
        lambda a: pd.Series(a).rank(pct=True).iloc[-1],
        raw=False
    )


def merge_indicator_with_carry(master_df, ind_name):
    ind = get_clean_indicator(ind_name).copy()
    merged = pd.merge_asof(
        master_df.sort_values("Date"),
        ind.sort_values("Date"),
        on="Date",
        direction="backward"
    )
    merged[ind_name] = merged[ind_name].ffill()
    return merged


# =========================================================
# FREQUENCY-AWARE SIGNALS
# =========================================================
def expanding_percentile_on_updates(series, min_periods_updates=24):
    s = pd.Series(series).copy()
    out = pd.Series(np.nan, index=s.index, dtype=float)

    valid_idx = s[s.notna()].index
    if len(valid_idx) == 0:
        return out

    ranked_vals = pd.Series(np.nan, index=valid_idx, dtype=float)
    vals_list = []

    for idx in valid_idx:
        vals_list.append(s.loc[idx])
        if len(vals_list) >= min_periods_updates:
            ranked_vals.loc[idx] = pd.Series(vals_list).rank(pct=True).iloc[-1]

    out.loc[valid_idx] = ranked_vals
    out = out.ffill()
    return out


def infer_update_gap_from_original_indicator(ind_name):
    ind = get_clean_indicator(ind_name).copy()
    ind = ind.dropna(subset=[ind_name]).sort_values("Date").reset_index(drop=True)
    if len(ind) < 5:
        return np.nan
    return ind["Date"].diff().dropna().dt.days.median()


def make_signal_frequency_aware_from_original(ind_name, merged_series, daily_window=252):
    s = pd.Series(merged_series).copy()

    if ind_name in BINARY_INDICATORS or s.dropna().nunique() <= 2:
        s = s.rolling(window=252, min_periods=63).mean()

    gap = infer_update_gap_from_original_indicator(ind_name)

    if pd.isna(gap) or gap <= 5:
        return trailing_percentile(s, window=daily_window, min_periods=63)
    elif gap <= 15:
        return trailing_percentile(s, window=252, min_periods=26)
    else:
        return expanding_percentile_on_updates(s, min_periods_updates=24)


# =========================================================
# EVENT-LEAD TARGET
# =========================================================
def build_realized_crisis_events(price_df, drawdown_thresh=DRAWDOWN_THRESH):
    df = ensure_date_col(price_df).copy()
    price_col = [c for c in df.columns if c != "Date"][0]
    df = df[["Date", price_col]].rename(columns={price_col: "close"}).sort_values("Date").reset_index(drop=True)

    df["rolling_peak"] = df["close"].cummax()
    df["drawdown"] = 1 - (df["close"] / df["rolling_peak"])

    prev_dd = df["drawdown"].shift(1).fillna(0)
    crisis_start = (prev_dd < drawdown_thresh) & (df["drawdown"] >= drawdown_thresh)

    events = df.loc[crisis_start, ["Date"]].copy().reset_index(drop=True)
    events["event_id"] = np.arange(1, len(events) + 1)
    return events


def make_event_lead_target(prices, dates, drawdown_thresh=DRAWDOWN_THRESH, lead_days=LEAD_DAYS):
    df = pd.DataFrame({
        "Date": pd.to_datetime(dates),
        "close": pd.Series(prices).reset_index(drop=True)
    }).sort_values("Date").reset_index(drop=True)

    events = build_realized_crisis_events(df[["Date", "close"]], drawdown_thresh=drawdown_thresh)
    event_dates = set(pd.to_datetime(events["Date"]))

    y = pd.Series(0.0, index=df.index)

    # mark all days in the lead window before each crisis start as positive
    for ev_date in pd.to_datetime(events["Date"]):
        ev_idx_list = df.index[df["Date"] == ev_date].tolist()
        if len(ev_idx_list) == 0:
            continue
        ev_idx = ev_idx_list[0]
        start_idx = max(0, ev_idx - lead_days)
        y.iloc[start_idx:ev_idx] = 1.0

    # optional: exclude crisis-start day itself from training target
    crisis_start_mask = df["Date"].isin(event_dates)
    y.loc[crisis_start_mask] = np.nan

    return y.values


# =========================================================
# TRIGGER ENGINE
# =========================================================
def extract_discrete_events(is_on, reset_days=RESET_DAYS):
    is_on = pd.Series(is_on).fillna(False).astype(bool)

    events = np.zeros(len(is_on), dtype=int)
    last_event_idx = -10**9
    prev_on = False

    for i in range(len(is_on)):
        if is_on.iloc[i]:
            if (not prev_on) or ((i - last_event_idx) >= reset_days):
                events[i] = 1
                last_event_idx = i
            prev_on = True
        else:
            prev_on = False

    return events


def fit_threshold_from_training(raw_signal_train, alert_threshold=ALERT_THRESHOLD, smooth_days=SMOOTH_DAYS):
    s = pd.Series(raw_signal_train, dtype=float)
    smoothed = s.rolling(smooth_days, min_periods=smooth_days).mean().dropna()
    if len(smoothed) == 0:
        return np.nan
    return np.nanpercentile(smoothed, 100 * (1 - alert_threshold))


def build_trigger_engine(raw_signal, threshold, smooth_days=SMOOTH_DAYS, reset_days=RESET_DAYS):
    s = pd.Series(raw_signal, dtype=float)
    smoothed = s.rolling(smooth_days, min_periods=smooth_days).mean()
    is_on = smoothed >= threshold
    events = extract_discrete_events(is_on, reset_days=reset_days)

    return pd.DataFrame({
        "raw_signal": s,
        "smoothed_signal": smoothed,
        "is_on": is_on,
        "event": events
    })


def choose_model(model_type="random_forest"):
    if model_type == "tree":
        return DecisionTreeClassifier(
            max_depth=TREE_MAX_DEPTH,
            min_samples_leaf=TREE_MIN_SAMPLES_LEAF,
            random_state=RANDOM_STATE,
            class_weight="balanced"
        )

    return RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )


def get_oof_train_probs(model_type, X_train, y_train, n_splits=INNER_OOF_SPLITS):
    X_train = X_train.reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)

    oof_prob = pd.Series(np.nan, index=X_train.index, dtype=float)

    # need enough rows to split and class variation
    if len(X_train) < (n_splits + 1) * 20 or y_train.nunique() < 2:
        return oof_prob.values

    tscv = TimeSeriesSplit(n_splits=n_splits)

    for inner_train_idx, inner_val_idx in tscv.split(X_train):
        X_tr = X_train.iloc[inner_train_idx]
        y_tr = y_train.iloc[inner_train_idx]
        X_val = X_train.iloc[inner_val_idx]

        if y_tr.nunique() < 2:
            continue

        model = choose_model(model_type)
        model.fit(X_tr, y_tr)
        oof_prob.iloc[inner_val_idx] = model.predict_proba(X_val)[:, 1]

    return oof_prob.values


# =========================================================
# BUILD DATA
# =========================================================
def build_master_ds_for_ml_fixed():
    spx = ensure_date_col(spx_c)
    price_col = [c for c in spx.columns if c != "Date"][0]
    spx = spx[["Date", price_col]].rename(columns={price_col: "close"}).sort_values("Date").reset_index(drop=True)

    spx["target_downturn"] = make_event_lead_target(
        prices=spx["close"].values,
        dates=spx["Date"].values,
        drawdown_thresh=DRAWDOWN_THRESH,
        lead_days=LEAD_DAYS
    )

    master_ds = spx[["Date", "close", "target_downturn"]].copy()

    for name in sorted(indicators.keys()):
        try:
            master_ds = merge_indicator_with_carry(master_ds, name)
            master_ds[f"{name}__pct"] = make_signal_frequency_aware_from_original(
                ind_name=name,
                merged_series=master_ds[name]
            )
        except Exception as e:
            print(f"Skipping {name}: {e}")

    return master_ds


def prepare_later_df_from_master(master_ds):
    full_feature_cols = [c for c in master_ds.columns if c.endswith("__pct")]

    later_df_full = master_ds.loc[master_ds["Date"] >= pd.Timestamp(MODEL_START_DATE)].copy()
    later_df_full = later_df_full.dropna(subset=["target_downturn"]).reset_index(drop=True)

    # still global coverage filter for simplicity; stricter anti-lookahead version would do this per train fold
    coverage = later_df_full[full_feature_cols].notna().mean().sort_values(ascending=False)
    keep_features = coverage[coverage >= (1 - MAX_MISSING_FEATURE_FRAC)].index.tolist()

    later_df = later_df_full[["Date", "close", "target_downturn"] + keep_features].copy()
    later_df[keep_features] = later_df[keep_features].ffill()

    return later_df, keep_features


def build_valid_folds(later_df, keep_features):
    first_test_start = WARMUP_DAYS + MIN_TRAIN_ROWS + PURGE_DAYS
    remaining = len(later_df) - first_test_start

    if remaining <= 0:
        raise ValueError("Not enough later-sample rows after warmup + minimum train + purge.")

    fold_size = remaining // N_FOLDS
    if fold_size < MIN_TEST_ROWS:
        raise ValueError("Fold size too small. Reduce N_FOLDS or MIN_TRAIN_ROWS.")

    valid_fold_data = []

    for i in range(N_FOLDS):
        test_start = first_test_start + i * fold_size
        test_end = first_test_start + (i + 1) * fold_size if i < N_FOLDS - 1 else len(later_df)
        train_end = test_start - PURGE_DAYS

        train_df = later_df.iloc[WARMUP_DAYS:train_end].copy().reset_index(drop=True)
        test_df = later_df.iloc[test_start:test_end].copy().reset_index(drop=True)

        if len(train_df) < MIN_TRAIN_ROWS or len(test_df) < MIN_TEST_ROWS:
            continue

        medians = train_df[keep_features].median()
        train_df[keep_features] = train_df[keep_features].fillna(medians)
        test_df[keep_features] = test_df[keep_features].fillna(medians)

        y_train = train_df["target_downturn"].astype(int)
        if y_train.nunique() < 2:
            continue

        valid_fold_data.append((i + 1, train_df, test_df))

    return valid_fold_data


def split_selection_and_evaluation_folds(valid_fold_data):
    if len(valid_fold_data) < 2:
        raise ValueError("Need at least 2 valid folds to separate selection and evaluation.")

    split_idx = max(1, len(valid_fold_data) // 2)
    selection_folds = valid_fold_data[:split_idx]
    evaluation_folds = valid_fold_data[split_idx:]

    if len(evaluation_folds) == 0:
        raise ValueError("No evaluation folds left after split.")

    return selection_folds, evaluation_folds


# =========================================================
# EVENT METRICS
# =========================================================
def get_event_starts_from_test_df(test_df):
    y_test_series = pd.Series(test_df["target_downturn"]).fillna(0).astype(int).reset_index(drop=True)
    event_starts = []
    prev = 0

    for i in range(len(y_test_series)):
        if y_test_series.iloc[i] == 1 and prev == 0:
            event_starts.append(test_df["Date"].iloc[i])
        prev = y_test_series.iloc[i]

    return pd.Series(event_starts, dtype="datetime64[ns]")


def compute_distinct_crisis_metrics(trigger_dates, realized_events, horizon_years=LEAD_DAYS / 252):
    trigger_dates = pd.to_datetime(pd.Series(trigger_dates)).dropna().sort_values().reset_index(drop=True)
    event_dates = pd.to_datetime(realized_events["Date"]).dropna().sort_values().reset_index(drop=True)

    used_trigger_idx = set()
    tp = 0
    lead_sum = 0.0

    for ev in event_dates:
        window_start = ev - pd.Timedelta(days=int(horizon_years * 365.25))

        eligible = [
            (idx, dt) for idx, dt in enumerate(trigger_dates)
            if idx not in used_trigger_idx and window_start <= dt < ev
        ]

        if len(eligible) == 0:
            continue

        idx, first = eligible[0]
        used_trigger_idx.add(idx)
        tp += 1
        lead_sum += (ev - first).days

    precision = tp / len(trigger_dates) if len(trigger_dates) else 0.0
    recall = tp / len(event_dates) if len(event_dates) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    avg_lead = lead_sum / tp if tp else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "avg_lead": avg_lead,
        "n_triggers": len(trigger_dates),
        "n_crises": len(event_dates),
        "n_caught": tp
    }


def compute_fold_alarm_metrics(test_df, test_prob, threshold):
    test_engine = build_trigger_engine(test_prob, threshold)
    trigger_dates = test_df.loc[test_engine["event"].values == 1, "Date"].reset_index(drop=True)

    # realized events within this fold's actual test window only
    fold_prices = test_df[["Date", "close"]].copy()
    realized_events = build_realized_crisis_events(fold_prices, drawdown_thresh=DRAWDOWN_THRESH)

    metrics = compute_distinct_crisis_metrics(
        trigger_dates=trigger_dates,
        realized_events=realized_events,
        horizon_years=LEAD_DAYS / 252
    )
    return metrics, trigger_dates


def get_test_date_span(valid_fold_data):
    all_test_dates = pd.concat(
        [test_df["Date"] for _, _, test_df in valid_fold_data],
        ignore_index=True
    ).sort_values().reset_index(drop=True)

    return all_test_dates.min(), all_test_dates.max()


def build_realized_crisis_events_in_eval_window(price_df, start_date, end_date, drawdown_thresh=DRAWDOWN_THRESH):
    events = build_realized_crisis_events(price_df, drawdown_thresh=drawdown_thresh)
    events = events.loc[
        (events["Date"] >= pd.Timestamp(start_date)) &
        (events["Date"] <= pd.Timestamp(end_date))
    ].reset_index(drop=True)
    events["event_id"] = np.arange(1, len(events) + 1)
    return events


# =========================================================
# SINGLE-FEATURE SEARCH ON SELECTION FOLDS
# =========================================================
def score_feature_on_folds(feat, fold_data):
    rows = []

    for fold_id, train_df, test_df in fold_data:
        X_train = train_df[[feat]].astype(float)
        X_test = test_df[[feat]].astype(float)

        y_train = train_df["target_downturn"].astype(int)
        if y_train.nunique() < 2:
            continue

        oof_train_prob = get_oof_train_probs(MODEL_TYPE, X_train, y_train, n_splits=INNER_OOF_SPLITS)
        threshold = fit_threshold_from_training(oof_train_prob)
        if pd.isna(threshold):
            continue

        model = choose_model(MODEL_TYPE)
        model.fit(X_train, y_train)
        test_prob = model.predict_proba(X_test)[:, 1]

        metrics, _ = compute_fold_alarm_metrics(test_df, test_prob, threshold)

        rows.append({
            "feature": feat,
            "indicator": feat.replace("__pct", ""),
            "fold": fold_id,
            "test_precision": metrics["precision"],
            "test_recall": metrics["recall"],
            "test_f1": metrics["f1"],
            "test_avg_lead": metrics["avg_lead"],
            "n_triggers": metrics["n_triggers"],
            "n_crises": metrics["n_crises"],
            "n_caught": metrics["n_caught"]
        })

    return rows


def build_single_summary_df(keep_features, selection_folds):
    single_rows = []

    for feat in keep_features:
        single_rows.extend(score_feature_on_folds(feat, selection_folds))

    single_results_df = pd.DataFrame(single_rows)

    if single_results_df.empty:
        raise ValueError("No valid single-feature results were produced.")

    single_summary_df = (
        single_results_df
        .groupby(["feature", "indicator"], as_index=False)
        .agg(
            mean_test_precision=("test_precision", "mean"),
            mean_test_recall=("test_recall", "mean"),
            mean_test_f1=("test_f1", "mean"),
            mean_test_avg_lead=("test_avg_lead", "mean"),
            mean_n_triggers=("n_triggers", "mean"),
            mean_n_crises=("n_crises", "mean"),
            mean_n_caught=("n_caught", "mean"),
            n_folds=("fold", "nunique")
        )
        .sort_values(["mean_test_f1", "mean_test_recall", "mean_test_precision"], ascending=False)
        .reset_index(drop=True)
    )

    return single_results_df, single_summary_df


# =========================================================
# COMBO SEARCH ON EVALUATION FOLDS
# =========================================================
def score_combo_on_folds(combo, fold_data):
    rows = []

    for fold_id, train_df, test_df in fold_data:
        X_train = train_df[list(combo)].astype(float)
        X_test = test_df[list(combo)].astype(float)

        y_train = train_df["target_downturn"].astype(int)
        if y_train.nunique() < 2:
            continue

        oof_train_prob = get_oof_train_probs(MODEL_TYPE, X_train, y_train, n_splits=INNER_OOF_SPLITS)
        threshold = fit_threshold_from_training(oof_train_prob)
        if pd.isna(threshold):
            continue

        model = choose_model(MODEL_TYPE)
        model.fit(X_train, y_train)
        test_prob = model.predict_proba(X_test)[:, 1]

        metrics, _ = compute_fold_alarm_metrics(test_df, test_prob, threshold)

        rows.append({
            "combo": " + ".join([c.replace("__pct", "") for c in combo]),
            "size": len(combo),
            "fold": fold_id,
            "test_precision": metrics["precision"],
            "test_recall": metrics["recall"],
            "test_f1": metrics["f1"],
            "test_avg_lead": metrics["avg_lead"],
            "n_triggers": metrics["n_triggers"],
            "n_crises": metrics["n_crises"],
            "n_caught": metrics["n_caught"]
        })

    return rows


def build_combo_summary_df(finalist_features, evaluation_folds):
    combo_fold_rows = []

    for k in range(1, min(MAX_COMBO_SIZE, len(finalist_features)) + 1):
        for combo in combinations(finalist_features, k):
            combo_fold_rows.extend(score_combo_on_folds(combo, evaluation_folds))

    combo_fold_results_df = pd.DataFrame(combo_fold_rows)

    if combo_fold_results_df.empty:
        raise ValueError("No valid combo results were produced.")

    combo_summary_df = (
        combo_fold_results_df
        .groupby("combo", as_index=False)
        .agg(
            size=("size", "first"),
            mean_test_precision=("test_precision", "mean"),
            mean_test_recall=("test_recall", "mean"),
            mean_test_f1=("test_f1", "mean"),
            mean_test_avg_lead=("test_avg_lead", "mean"),
            mean_n_triggers=("n_triggers", "mean"),
            mean_n_crises=("n_crises", "mean"),
            mean_n_caught=("n_caught", "mean"),
            n_folds=("fold", "nunique")
        )
        .query("n_folds >= @MIN_FOLDS_REQUIRED")
        .sort_values(["mean_test_f1", "mean_test_recall", "mean_test_precision"], ascending=False)
        .reset_index(drop=True)
    )

    return combo_fold_results_df, combo_summary_df


# =========================================================
# FULL SEARCH PIPELINE
# =========================================================
def run_ml_combo_search_corrected():
    master_ds = build_master_ds_for_ml_fixed()
    later_df, keep_features = prepare_later_df_from_master(master_ds)
    valid_fold_data = build_valid_folds(later_df, keep_features)

    selection_folds, evaluation_folds = split_selection_and_evaluation_folds(valid_fold_data)

    single_results_df, single_summary_df = build_single_summary_df(keep_features, selection_folds)

    finalist_features = single_summary_df.head(N_FINALISTS)["feature"].tolist()
    combo_fold_results_df, combo_summary_df = build_combo_summary_df(finalist_features, evaluation_folds)

    eval_start, eval_end = get_test_date_span(evaluation_folds)
    realized_events_eval = build_realized_crisis_events_in_eval_window(
        spx_c,
        start_date=eval_start,
        end_date=eval_end,
        drawdown_thresh=DRAWDOWN_THRESH
    )

    return {
        "master_ds": master_ds,
        "later_df": later_df,
        "keep_features": keep_features,
        "valid_fold_data": valid_fold_data,
        "selection_folds": selection_folds,
        "evaluation_folds": evaluation_folds,
        "single_results": single_results_df,
        "single_summary": single_summary_df,
        "finalist_features": finalist_features,
        "combo_fold_results": combo_fold_results_df,
        "combo_summary": combo_summary_df,
        "realized_events_eval": realized_events_eval,
        "eval_start": eval_start,
        "eval_end": eval_end
    }


# =========================================================
# TRIGGERS FOR A COMBO
# =========================================================
def combo_name_to_feature_list(combo_name):
    return [f"{x.strip()}__pct" for x in combo_name.split(" + ")]


def get_trigger_dates_for_combo(combo_name, fold_data):
    feats = combo_name_to_feature_list(combo_name)
    all_trigger_dates = []

    for fold_id, train_df, test_df in fold_data:
        if any(f not in train_df.columns for f in feats):
            continue

        X_train = train_df[feats].astype(float)
        X_test = test_df[feats].astype(float)
        y_train = train_df["target_downturn"].astype(int)

        if y_train.nunique() < 2:
            continue

        oof_train_prob = get_oof_train_probs(MODEL_TYPE, X_train, y_train, n_splits=INNER_OOF_SPLITS)
        threshold = fit_threshold_from_training(oof_train_prob)
        if pd.isna(threshold):
            continue

        model = choose_model(MODEL_TYPE)
        model.fit(X_train, y_train)

        test_prob = model.predict_proba(X_test)[:, 1]
        test_engine = build_trigger_engine(test_prob, threshold)
        fold_trigger_dates = test_df.loc[test_engine["event"].values == 1, "Date"].reset_index(drop=True)
        all_trigger_dates.append(pd.to_datetime(fold_trigger_dates))

    if len(all_trigger_dates) == 0:
        return pd.Series([], dtype="datetime64[ns]")

    return (
        pd.concat(all_trigger_dates, ignore_index=True)
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )


# =========================================================
# TABLE 1: INDIVIDUAL COMBOS
# =========================================================
def build_individual_combo_table(combo_summary_df, evaluation_folds, realized_events_eval, top_n=12):
    top_combos = combo_summary_df.head(top_n)["combo"].tolist()
    rows = []

    for combo_name in top_combos:
        trigger_dates = get_trigger_dates_for_combo(combo_name, evaluation_folds)
        m = compute_distinct_crisis_metrics(
            trigger_dates=trigger_dates,
            realized_events=realized_events_eval,
            horizon_years=LEAD_DAYS / 252
        )

        rows.append({
            "combo": combo_name,
            "precision": m["precision"],
            "recall": m["recall"],
            "f1": m["f1"],
            "avg_lead": m["avg_lead"],
            "n_triggers": m["n_triggers"],
            "n_crises": m["n_crises"],
            "n_caught": m["n_caught"]
        })

    return pd.DataFrame(rows).sort_values(
        ["recall", "f1", "precision", "avg_lead"],
        ascending=False
    ).reset_index(drop=True)


# =========================================================
# TABLE 2: AUGMENTED COMBOS
# =========================================================
def build_augmented_combo_table(combo_summary_df, evaluation_folds, realized_events_eval, keep_features, top_n=12):
    top_combos = combo_summary_df.head(top_n)["combo"].tolist()
    keep_feature_names = {c.replace("__pct", "") for c in keep_features}
    available_augmenters = [x for x in REGIME_AUGMENT_CANDIDATES if x in keep_feature_names]

    rows = []

    for base_combo in top_combos:
        base_triggers = get_trigger_dates_for_combo(base_combo, evaluation_folds)
        base_parts = set([x.strip() for x in base_combo.split(" + ")])

        for aug_indicator in available_augmenters:
            if aug_indicator in base_parts:
                continue

            aug_triggers = get_trigger_dates_for_combo(aug_indicator, evaluation_folds)

            union_triggers = (
                pd.concat([pd.Series(base_triggers), pd.Series(aug_triggers)], ignore_index=True)
                .dropna()
                .drop_duplicates()
                .sort_values()
                .reset_index(drop=True)
            )

            m = compute_distinct_crisis_metrics(
                trigger_dates=union_triggers,
                realized_events=realized_events_eval,
                horizon_years=LEAD_DAYS / 252
            )

            rows.append({
                "base_combo": base_combo,
                "augmenter": aug_indicator,
                "augmented_combo": f"{base_combo} || {aug_indicator}",
                "precision": m["precision"],
                "recall": m["recall"],
                "f1": m["f1"],
                "avg_lead": m["avg_lead"],
                "n_triggers": m["n_triggers"],
                "n_crises": m["n_crises"],
                "n_caught": m["n_caught"]
            })

    return pd.DataFrame(rows).sort_values(
        ["recall", "f1", "precision", "avg_lead"],
        ascending=False
    ).reset_index(drop=True)


# =========================================================
# RUN
# =========================================================
results_fixed = run_ml_combo_search_corrected()

individual_combo_table = build_individual_combo_table(
    combo_summary_df=results_fixed["combo_summary"],
    evaluation_folds=results_fixed["evaluation_folds"],
    realized_events_eval=results_fixed["realized_events_eval"],
    top_n=12
)

augmented_combo_table = build_augmented_combo_table(
    combo_summary_df=results_fixed["combo_summary"],
    evaluation_folds=results_fixed["evaluation_folds"],
    realized_events_eval=results_fixed["realized_events_eval"],
    keep_features=results_fixed["keep_features"],
    top_n=12
)

print(f"\nEvaluation window: {results_fixed['eval_start'].date()} to {results_fixed['eval_end'].date()}")
print(f"Number of realized crises in evaluation window: {len(results_fixed['realized_events_eval'])}")

print("\n=== TOP SINGLE INDICATORS (selection folds only) ===")
display(
    results_fixed["single_summary"].head(20).style.background_gradient(
        subset=["mean_test_precision", "mean_test_recall", "mean_test_f1", "mean_test_avg_lead", "mean_n_caught"],
        cmap="RdYlGn"
    )
)

print("\n=== INDIVIDUAL COMBOS (evaluation folds only) ===")
display(
    individual_combo_table.style.background_gradient(
        subset=["precision", "recall", "f1", "avg_lead", "n_caught"],
        cmap="RdYlGn"
    )
)

print("\n=== AUGMENTED COMBOS (evaluation folds only) ===")
display(
    augmented_combo_table.style.background_gradient(
        subset=["precision", "recall", "f1", "avg_lead", "n_caught"],
        cmap="RdYlGn"
    )
)


Evaluation window: 2012-02-03 to 2026-03-06
Number of realized crises in evaluation window: 23

=== TOP SINGLE INDICATORS (selection folds only) ===


,feature,indicator,mean_test_precision,mean_test_recall,mean_test_f1,mean_test_avg_lead,mean_n_triggers,mean_n_crises,mean_n_caught,n_folds
0,CBOE SKEW__pct,CBOE SKEW,0.666667,0.769231,0.714286,259.300000,15.000000,13.000000,10.000000,1
1,MACD__pct,MACD,0.533333,0.615385,0.571429,190.000000,15.000000,13.000000,8.000000,1
2,RSI (14)__pct,RSI (14),0.500000,0.615385,0.551724,118.375000,16.000000,13.000000,8.000000,1
3,Buffet Indicator__pct,Buffet Indicator,1.000000,0.307692,0.470588,200.500000,4.000000,13.000000,4.000000,1
4,AAII Bullish__pct,AAII Bullish,0.363636,0.615385,0.457143,112.375000,22.000000,13.000000,8.000000,1
5,AAII Bearish__pct,AAII Bearish,0.428571,0.461538,0.444444,83.166667,14.000000,13.000000,6.000000,1
6,AAII Bull-Bear Spread__pct,AAII Bull-Bear Spread,0.400000,0.461538,0.428571,69.500000,15.000000,13.000000,6.000000,1
7,Financial Stress (StL)__pct,Financial Stress (StL),0.333333,0.461538,0.387097,222.333333,18.000000,13.000000,6.000000,1
8,GDP Recession Indicator__pct,GDP Recession Indicator,0.600000,0.230769,0.333333,191.666667,5.000000,13.000000,3.000000,1
9,Equity Risk Premium__pct,Equity Risk Premium,0.333333,0.230769,0.272727,75.666667,9.000000,13.000000,3.000000,1



=== INDIVIDUAL COMBOS (evaluation folds only) ===


,combo,precision,recall,f1,avg_lead,n_triggers,n_crises,n_caught
0,RSI (14) + AAII Bull-Bear Spread,0.484848,0.695652,0.571429,151.250000,33,23,16
1,MACD + AAII Bull-Bear Spread,0.600000,0.652174,0.625000,155.133333,25,23,15
2,MACD + RSI (14) + AAII Bull-Bear Spread,0.483871,0.652174,0.555556,155.933333,31,23,15
3,MACD + AAII Bearish + AAII Bull-Bear Spread,0.441176,0.652174,0.526316,157.866667,34,23,15
4,AAII Bullish + AAII Bearish + Financial Stress (StL),0.428571,0.652174,0.517241,164.333333,35,23,15
5,RSI (14) + AAII Bull-Bear Spread + Financial Stress (StL),0.424242,0.608696,0.500000,167.714286,33,23,14
6,RSI (14) + Buffet Indicator + BB Lower,0.254545,0.608696,0.358974,134.357143,55,23,14
7,CBOE SKEW + AAII Bull-Bear Spread + Financial Stress (StL),0.565217,0.565217,0.565217,122.846154,23,23,13
8,CBOE SKEW + AAII Bullish + Financial Stress (StL),0.523810,0.478261,0.500000,100.090909,21,23,11
9,CBOE SKEW + RSI (14) + AAII Bearish,0.666667,0.434783,0.526316,86.300000,15,23,10



=== AUGMENTED COMBOS (evaluation folds only) ===


,base_combo,augmenter,augmented_combo,precision,recall,f1,avg_lead,n_triggers,n_crises,n_caught
0,RSI (14) + AAII Bull-Bear Spread,PS Ratio,RSI (14) + AAII Bull-Bear Spread || PS Ratio,0.295775,0.913043,0.446809,229.619048,71,23,21
1,RSI (14) + AAII Bull-Bear Spread,Shiller CAPE,RSI (14) + AAII Bull-Bear Spread || Shiller CAPE,0.139073,0.913043,0.241379,222.476190,151,23,21
2,MACD + Equity Risk Premium,PS Ratio,MACD + Equity Risk Premium || PS Ratio,0.425532,0.869565,0.571429,234.150000,47,23,20
3,CBOE SKEW + MACD + RSI (14),PS Ratio,CBOE SKEW + MACD + RSI (14) || PS Ratio,0.377358,0.869565,0.526316,235.450000,53,23,20
4,CBOE SKEW + RSI (14) + AAII Bearish,PS Ratio,CBOE SKEW + RSI (14) + AAII Bearish || PS Ratio,0.370370,0.869565,0.519481,225.300000,54,23,20
5,CBOE SKEW + AAII Bull-Bear Spread + Financial Stress (StL),PS Ratio,CBOE SKEW + AAII Bull-Bear Spread + Financial Stress (StL) || PS Ratio,0.333333,0.869565,0.481928,226.400000,60,23,20
6,CBOE SKEW + AAII Bullish + Financial Stress (StL),PS Ratio,CBOE SKEW + AAII Bullish + Financial Stress (StL) || PS Ratio,0.333333,0.869565,0.481928,216.400000,60,23,20
7,MACD + AAII Bull-Bear Spread,PS Ratio,MACD + AAII Bull-Bear Spread || PS Ratio,0.312500,0.869565,0.459770,238.700000,64,23,20
8,MACD + RSI (14) + AAII Bull-Bear Spread,PS Ratio,MACD + RSI (14) + AAII Bull-Bear Spread || PS Ratio,0.294118,0.869565,0.439560,235.650000,68,23,20
9,RSI (14) + AAII Bull-Bear Spread + Financial Stress (StL),PS Ratio,RSI (14) + AAII Bull-Bear Spread + Financial Stress (StL) || PS Ratio,0.281690,0.869565,0.425532,238.550000,71,23,20
